## Step 1: 安裝套件

首先安裝必要的 Python 套件。在 Google Colab 上執行以下指令。

> **說明**：
> - `transformers` — HuggingFace 提供的預訓練模型庫
> - `torch` — PyTorch 深度學習框架
> - `scikit-learn` — 評估指標計算
> - `pandas` / `matplotlib` — 資料分析與視覺化

In [ ]:
# 安裝必要套件（Colab 環境）
!pip install transformers torch scikit-learn pandas matplotlib seaborn -q
print("套件安裝完成！")

套件安裝完成！


In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# 中文字型設定（Colab）
matplotlib.rcParams['font.family'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("所有套件載入完成！")
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")

所有套件載入完成！
PyTorch 版本: 2.11.0+cu128
CUDA 可用: True


In [ ]:
import torch

print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : NVIDIA A100-SXM4-80GB


In [ ]:
import torch
import transformers
import accelerate
import pandas
import numpy
import sklearn
import matplotlib
import seaborn
import tqdm
import sentencepiece

print(f"PyTorch          : {torch.__version__}")
print(f"Transformers     : {transformers.__version__}")
print(f"Accelerate       : {accelerate.__version__}")
print(f"Pandas           : {pandas.__version__}")
print(f"NumPy            : {numpy.__version__}")
print(f"Scikit-learn     : {sklearn.__version__}")
print(f"Matplotlib       : {matplotlib.__version__}")
print(f"Seaborn          : {seaborn.__version__}")
print(f"Tqdm             : {tqdm.__version__}")
print(f"SentencePiece    : {sentencepiece.__version__}")

PyTorch          : 2.11.0+cu128
Transformers     : 5.12.0
Accelerate       : 1.14.0
Pandas           : 2.2.2
NumPy            : 2.0.2
Scikit-learn     : 1.6.1
Matplotlib       : 3.10.0
Seaborn          : 0.13.2
Tqdm             : 4.67.3
SentencePiece    : 0.2.1


In [ ]:
import transformers
print(transformers.__version__)

5.12.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: 設定超參數與標籤對應

在開始之前，我們先定義所有重要的超參數與標籤映射表。

### 重要參數說明
- **MODEL_NAME**: 使用 `bert-base-chinese`，這是 Google 發布的中文 BERT 模型
- **MAX_LEN**: 輸入文字的最大 Token 長度（256）
- **BATCH_SIZE**: 每次訓練的樣本數（16）
- **EPOCHS**: 訓練完整資料集的次數（5）
- **LR**: 學習率，建議 BERT fine-tuning 使用 2e-5

### 為什麼用 bert-base-chinese？
BERT（Bidirectional Encoder Representations from Transformers）是一個雙向 Transformer 模型。
中文版本在大量中文語料上預訓練，能夠理解中文文字的語意。

In [ ]:
import torch
# 定義全域 device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前偵測到的執行裝置: {device}")

目前偵測到的執行裝置: cuda


In [ ]:
# ============================================================
# 超參數設定
# ============================================================
MODEL_NAME = "bert-base-chinese"
MAX_LEN = 512 #從256改384再改512
BATCH_SIZE = 8    # 資料集較小（1000筆），使用小 batch size
EPOCHS = 12       # 資料少，多訓練幾個 epoch
LR = 2e-5

# 四個預測欄位及其標籤（依照競賽官方規格定義）
EVAL_FIELDS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years", "N/A"],
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"],
}

# 各欄位的評分權重
FIELD_WEIGHTS = {
    "promise_status": 0.20,
    "evidence_status": 0.30,
    "evidence_quality": 0.35,
    "verification_timeline": 0.15,
}

# 建立標籤 ↔ ID 的對應表
label2id = {field: {lab: i for i, lab in enumerate(labels)} for field, labels in EVAL_FIELDS.items()}
id2label = {field: {i: lab for i, lab in enumerate(labels)} for field, labels in EVAL_FIELDS.items()}
num_labels = {field: len(labels) for field, labels in EVAL_FIELDS.items()}

print("超參數設定完成！")
print("\n標籤對應表：")
for field, mapping in label2id.items():
    print(f"  {field}: {mapping}")

超參數設定完成！

標籤對應表：
  promise_status: {'No': 0, 'Yes': 1}
  verification_timeline: {'already': 0, 'within_2_years': 1, 'between_2_and_5_years': 2, 'more_than_5_years': 3, 'N/A': 4}
  evidence_status: {'Yes': 0, 'No': 1, 'N/A': 2}
  evidence_quality: {'Clear': 0, 'Not Clear': 1, 'Misleading': 2, 'N/A': 3}


In [ ]:
# ============================================================
# 三專家任務設定 (Expert Branching Configuration)
# ============================================================

# 1. ES + EQ 聯軍：共享雙重池化細節特徵
EXPERT_QUALITY_FIELDS = {
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"],
}

# 2. VT 獨立：專攻 Focal Loss 與極端不平衡樣本
EXPERT_TIMELINE_FIELDS = {
    "verification_timeline": ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years", "N/A"],
}

# 3. PS 獨立：穩住大盤基礎
EXPERT_ANCHOR_FIELDS = {
    "promise_status": ["No", "Yes"],
}

# 合併所有欄位以方便後續處理
ALL_EXPERT_FIELDS = {**EXPERT_QUALITY_FIELDS, **EXPERT_TIMELINE_FIELDS, **EXPERT_ANCHOR_FIELDS}

# 建立 label2id, id2label, num_labels
expert_label2id = {
    field: {lab: i for i, lab in enumerate(labels)}
    for field, labels in ALL_EXPERT_FIELDS.items()
}
expert_id2label = {
    field: {i: lab for i, lab in enumerate(labels)}
    for field, labels in ALL_EXPERT_FIELDS.items()
}
expert_num_labels = {
    field: len(labels)
    for field, labels in ALL_EXPERT_FIELDS.items()
}

print("\n三專家任務設定完成！")
print("🔥 Quality Expert (ES+EQ):", list(EXPERT_QUALITY_FIELDS.keys()))
print("⚪ Timeline Expert (VT):", list(EXPERT_TIMELINE_FIELDS.keys()))
print("⚪ Anchor Expert (PS):", list(EXPERT_ANCHOR_FIELDS.keys()))


三專家任務設定完成！
🔥 Quality Expert (ES+EQ): ['evidence_status', 'evidence_quality']
⚪ Timeline Expert (VT): ['verification_timeline']
⚪ Anchor Expert (PS): ['promise_status']


## Step 3: 載入資料
載入剛才下載的 JSON 檔案，並切分為訓練集（80%）與驗證集（20%）。

> **說明**：這份資料共 1,000 筆，來自台灣上市公司的 ESG 永續報告書。

## 用stratified split改進

In [ ]:
import json
import pandas as pd

# =====================================================
# 讀取 train
# =====================================================

with open(
    "/content/drive/MyDrive/AI_CUP_ESG/vpesg4k_train_1000_more.json",
    "r",
    encoding="utf-8"
) as f:
    train_data = json.load(f)

# =====================================================
# 讀取 val
# =====================================================

with open(
    "/content/drive/MyDrive/AI_CUP_ESG/vpesg4k_val_1000.json",
    "r",
    encoding="utf-8"
) as f:
    val_data = json.load(f)

# =====================================================
# 合併
# =====================================================

all_data = train_data + val_data

print("Train:", len(train_data))
print("Val  :", len(val_data))
print("Total:", len(all_data))

# =====================================================
# 將空值統一轉成 N/A
# =====================================================

for item in all_data:

    for key, value in item.items():

        # None
        if value is None:
            item[key] = "N/A"
            continue

        # NaN
        if pd.isna(value):
            item[key] = "N/A"
            continue

        # 空字串
        if isinstance(value, str):

            if value.strip() == "":
                item[key] = "N/A"

# =====================================================
# 儲存
# =====================================================

save_path = (
    "/content/drive/MyDrive/AI_CUP_ESG/"
    "vpesg4k_train_2000.json"
)

with open(
    save_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        all_data,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"\n✅ 已儲存：{save_path}")
print(f"總筆數：{len(all_data)}")

Train: 1000
Val  : 1000
Total: 2000

✅ 已儲存：/content/drive/MyDrive/AI_CUP_ESG/vpesg4k_train_2000.json
總筆數：2000


In [ ]:
import json
from pathlib import Path

# =====================================================
# Read merged training data
# =====================================================

DATA_DIR = Path("data")
TRAIN_JSON = DATA_DIR / "vpesg4k_train_2000.json"

with open(
    TRAIN_JSON,
    "r",
    encoding="utf-8"
) as f:
    all_data = json.load(f)

print(f"Total training samples: {len(all_data)}")

## Feature Engineering




PyTorch 的 `Dataset` 類別負責：
1. 儲存資料
2. 將文字 **Tokenize**（轉換成 BERT 能理解的 Token ID）
3. 將標籤文字轉換成數字 ID

### Tokenization 是什麼？
BERT 使用 WordPiece tokenizer 將文字切成子詞單元：
- 輸入: `"台灣水泥承諾2030年碳中和"`
- 輸出: `[101, 1921, 3968, 3717, 3686, 8811, 2030, 2399, 4988, 102, ...]`
  - `101` = [CLS] 開始符號
  - `102` = [SEP] 結束符號

### Padding & Truncation
所有輸入必須等長（MAX_LEN=256），所以：
- 短文字 → 補 0（padding）
- 長文字 → 截斷（truncation）

In [ ]:
#三專家
import torch
from torch.utils.data import Dataset
import re

# ===== 你的原始特徵字典（維持不變） =====
VAGUE_WORDS = ["持續", "持續關注", "推動", "強化", "提升", "改善", "逐步", "相關資訊", "致力於"]
CLEAR_KEYWORDS = ["溫室氣體", "再生能源", "資訊安全", "風險管理", "億元", "ISO", "查證", "盤查", "導入", "建置", "達成"]
LONG_TERM_WORDS = ["長期目標", "淨零", "淨零碳排", "巴黎協定", "2030", "2050"]
ALREADY_WORDS = ["截至", "已完成", "已達成", "本年度", "已建置", "已導入"]
SHORT_TERM_WORDS = ["2025", "2026", "未來兩年", "兩年內", "短期內", "近期"]

# ===== [核心改進：動態證據提取器] =====
def extract_raw_evidence(text):
    """直接從文本抓取具體的數字、單位與標準"""
    patterns = r'(\d+(\.\d+)?%|ISO\s*\d+|\d{4}年|\d+\s*(?:公噸|公斤|度|kWh|MW|億元|萬元))'
    matches = re.findall(patterns, str(text))
    found = [m[0] if isinstance(m, tuple) else m for m in matches]
    return "/".join(found[:3]) if found else "None"

def build_feature_prefix_old(item):
    text = item["data"]

    # 1. 你的基本特徵（布林值判斷）
    has_number = int(bool(re.search(r"\d", text)))
    has_year = int(bool(re.search(r"20\d{2}", text)))
    has_percent = int(bool(re.search(r"%|％", text)))

    # 2. 你的語義計數
    vague_count = sum(w in text for w in VAGUE_WORDS)
    clear_count = sum(w in text for w in CLEAR_KEYWORDS)

    # 3. 你的時間軸 Flag
    long_term_flag = int(any(w in text for w in LONG_TERM_WORDS))
    already_flag = int(any(w in text for w in ALREADY_WORDS))
    short_term_flag = int(any(w in text for w in SHORT_TERM_WORDS))

    # 4. [新增] 具體證據內容提取 (讓模型直接看數據)
    raw_evidence = extract_raw_evidence(text)

    # 5. ESG 類型
    esg_type = item.get("esg_type", "UNK")

    # 組合最強 Prefix
    prefix = (
        f"[ESG={esg_type}] "
        f"[NUM={has_number}] "
        f"[YEAR={has_year}] "
        f"[PERCENT={has_percent}] "
        f"[VAGUE={vague_count}] "
        f"[CLEAR={clear_count}] "
        f"[TIMELINE={already_flag}/{short_term_flag}/{long_term_flag}] " # 整合時間軸
        f"[RAW_EVD={raw_evidence}] " # <--- 這是救 Quality 的關鍵
    )

    return prefix

class ESGDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, fields, mode="train"):
        """
        mode:
          - "anchor": 訓練 PS 分支
          - "quality": 訓練 ES + EQ 分支
          - "timeline": 訓練 VT 分支
          - "inference": 最終推理
        """
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.fields = fields  # 此處傳入該專家負責的欄位，例如 ["evidence_status", "evidence_quality"]
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def build_text(self, item):
        # 1. 取得你寫的最強 Feature Prefix (包含 RAW_EVD)
        # 這對 ES + EQ 特別重要，因為 ES 會去「借用」這裡的數字資訊
        base_prefix = build_feature_prefix_old(item)

        # 2. 原始文字內容
        main_text = item["data"]

        # 3. [邏輯調整] 只有在最終推理或特定實驗時，才注入其他專家的結果
        # 在三專家平行訓練時，我們希望各分支獨立，避免錯誤傳導
        # 如果你正在做「實驗四：注入預測狀態」，才開啟下面這段
        extra_info = ""
        if "pred_promise_status" in item and self.mode == "quality":
            ps = item.get("pred_promise_status", "UNK")
            extra_info = f"[STATE: PS={ps}] "

        return extra_info + base_prefix + main_text

    def __getitem__(self, idx):
        item = self.data[idx]
        text = self.build_text(item)

        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}
        # 這裡只抓取該專家負責的 labels
        for field in self.fields:
            mapping = self.label2id[field]
            if field in item:
                label = item[field]

                if field == "verification_timeline":
                    if label == "longer_than_5_years":
                        label = "more_than_5_years"

                labels[field] = mapping[label]
            else:
                labels[field] = -1

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }

# collate_fn_factory 維持原樣即可，它是兼容的
def collate_fn_factory(fields):
    def collate_fn(batch):
        input_ids = torch.stack([b["input_ids"] for b in batch])
        attention_mask = torch.stack([b["attention_mask"] for b in batch])
        labels = {
            field: torch.tensor(
                [b["labels"][field] for b in batch],
                dtype=torch.long
            )
            for field in fields
        }
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
    return collate_fn

##Timeline用的featurev2


In [ ]:
# ============================================================
# 三專家ESG Feature Engineering (Final Version)
# Timeline / Quality 強化版
# ============================================================

import re
import torch
from torch.utils.data import Dataset

# ============================================================
# 關鍵詞字典
# ============================================================

VAGUE_WORDS = [
    "持續", "持續關注", "推動", "強化", "提升",
    "改善", "逐步", "相關資訊", "致力於",
    "精進", "落實", "優化", "深化", "促進"
]

CLEAR_KEYWORDS = [
    "溫室氣體", "再生能源", "資訊安全", "風險管理",
    "億元", "ISO", "查證", "盤查", "導入",
    "建置", "達成", "完成", "取得", "認證",
    "第三方", "確信"
]

# ============================================================
# Timeline 關鍵詞
# ============================================================

ALREADY_WORDS = [
    "截至", "已完成", "已達成",
    "本年度", "已建置", "已導入",
    "已取得", "完成查證"
]

SHORT_TERM_WORDS = [
    "2025", "2026",
    "未來兩年",
    "兩年內",
    "短期內",
    "近期"
]

MID_TERM_WORDS = [
    "2027", "2028", "2029", "2030",
    "五年內",
    "中期"
]

LONG_TERM_WORDS = [
    "2040", "2050",
    "長期目標",
    "淨零",
    "淨零碳排",
    "巴黎協定"
]

# ============================================================
# RAW Evidence 抽取
# ============================================================

def extract_raw_evidence(text):

    patterns = (
        r'(\d+(\.\d+)?%|'
        r'ISO\s*\d+|'
        r'\d{4}年|'
        r'\d+\s*(?:公噸|公斤|度|kWh|MW|億元|萬元))'
    )

    matches = re.findall(patterns, str(text))

    found = [
        m[0] if isinstance(m, tuple) else m
        for m in matches
    ]

    return "/".join(found[:3]) if found else "None"


# ============================================================
# Feature Prefix
# ============================================================

def build_feature_prefix(item):

    text = str(item["data"])

    # --------------------------------------------------------
    # 基本數值特徵
    # --------------------------------------------------------

    has_number = int(bool(re.search(r"\d", text)))

    has_year = int(bool(re.search(r"20\d{2}", text)))

    has_percent = int(bool(re.search(r"%|％", text)))

    # --------------------------------------------------------
    # 語義計數
    # --------------------------------------------------------

    vague_count = sum(text.count(w) for w in VAGUE_WORDS)

    clear_count = sum(text.count(w) for w in CLEAR_KEYWORDS)

    vague_high = int(vague_count >= 3)

    # --------------------------------------------------------
    # Timeline Features
    # --------------------------------------------------------

    already_flag = int(any(
        w in text for w in ALREADY_WORDS
    ))

    short_term_flag = int(any(
        w in text for w in SHORT_TERM_WORDS
    ))

    mid_term_flag = int(any(
        w in text for w in MID_TERM_WORDS
    ))

    long_term_flag = int(any(
        w in text for w in LONG_TERM_WORDS
    ))

    # --------------------------------------------------------
    # RAW Evidence
    # --------------------------------------------------------

    raw_evidence = extract_raw_evidence(text)

    # --------------------------------------------------------
    # ESG Type
    # --------------------------------------------------------

    esg_type = item.get("esg_type", "UNK")

    # --------------------------------------------------------
    # Prefix 組合
    # --------------------------------------------------------

    prefix = (

        f"[ESG={esg_type}] "

        f"[NUM={has_number}] "

        f"[YEAR={has_year}] "

        f"[PERCENT={has_percent}] "

        f"[VAGUE={vague_count}] "

        f"[VAGUE_HIGH={vague_high}] "

        f"[CLEAR={clear_count}] "

        f"[VT_ALREADY={already_flag}] "

        f"[VT_SHORT={short_term_flag}] "

        f"[VT_MID={mid_term_flag}] "

        f"[VT_LONG={long_term_flag}] "

        f"[RAW_EVD={raw_evidence}] "
    )

    return prefix


# ============================================================
# ESG Dataset
# ============================================================

class ESGDataset_featurev2(Dataset):

    def __init__(
        self,
        data,
        tokenizer,
        label2id,
        fields,
        mode="train"
    ):

        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.fields = fields
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def build_text(self, item):

        # Feature Prefix
        base_prefix = build_feature_prefix(item)

        # 原始文字
        main_text = item["data"]

        # 可選：注入 PS 狀態
        extra_info = ""

        if (
            "pred_promise_status" in item
            and self.mode == "quality"
        ):

            ps = item.get("pred_promise_status", "UNK")

            extra_info = f"[STATE: PS={ps}] "

        return extra_info + base_prefix + main_text

    def __getitem__(self, idx):

        item = self.data[idx]

        text = self.build_text(item)

        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}

        for field in self.fields:

            mapping = self.label2id[field]

            if field in item:
                label = item[field]

                if field == "verification_timeline":
                    if label == "longer_than_5_years":
                        label = "more_than_5_years"

                labels[field] = mapping[label]
            else:
                labels[field] = -1

        return {
            "input_ids": encoding["input_ids"].squeeze(0),

            "attention_mask": encoding["attention_mask"].squeeze(0),

            "labels": labels,
        }


# ============================================================
# Collate Function
# ============================================================

def collate_fn_factory(fields):

    def collate_fn(batch):

        input_ids = torch.stack([
            b["input_ids"] for b in batch
        ])

        attention_mask = torch.stack([
            b["attention_mask"] for b in batch
        ])

        labels = {

            field: torch.tensor(
                [b["labels"][field] for b in batch],
                dtype=torch.long
            )

            for field in fields
        }

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

    return collate_fn

In [ ]:
# =====================================================
# Tokenizer Example
# =====================================================

print("Tokenizer Example")
print("=" * 50)

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
print(f"Loaded tokenizer: {MODEL_NAME}")

# 取第一筆資料作為範例
sample_text = all_data[0]["data"][:50]

print(f"\nOriginal Text:\n{sample_text}")

# Tokenize
tokens = tokenizer.tokenize(sample_text)

print(f"\nTokens:\n{tokens}")

# Encoding
encoding = tokenizer(
    sample_text,
    max_length=20,
    padding="max_length",
    truncation=True
)

print(f"\nInput IDs:")
print(encoding["input_ids"])

print(f"\nAttention Mask:")
print(encoding["attention_mask"])

Tokenizer Example
Loaded tokenizer: bert-base-chinese

Original Text:
聯發科技除在「工作規則」中依照勞基法明確規定「員工在產假期間公司不得終止勞動契約」外，為支持同仁與其

Tokens:
['聯', '發', '科', '技', '除', '在', '「', '工', '作', '規', '則', '」', '中', '依', '照', '勞', '基', '法', '明', '確', '規', '定', '「', '員', '工', '在', '產', '假', '期', '間', '公', '司', '不', '得', '終', '止', '勞', '動', '契', '約', '」', '外', '，', '為', '支', '持', '同', '仁', '與', '其']

Input IDs:
[101, 5474, 4634, 4906, 2825, 7370, 1762, 519, 2339, 868, 6211, 1179, 520, 704, 898, 4212, 1246, 1825, 3791, 102]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Step 6: 模型架構設計



In [ ]:
# ============================================================
# 定義標籤數量 (解決 NameError: expert_label_counts)
# ============================================================

# 請確保你的 expert_label2id 已經定義好
# 這裡會根據你定義的映射表，自動計算每個欄位有多少類別
expert_label_counts = {
    field: len(labels) for field, labels in expert_label2id.items()
}

# 檢查一下數量是否正確
print("各欄位類別數量:", expert_label_counts)


各欄位類別數量: {'evidence_status': 3, 'evidence_quality': 4, 'verification_timeline': 5, 'promise_status': 2}


In [ ]:
#ES/EQ projector 分離 + EQ 輕等深度 + 不用太重 residual。
import torch
import torch.nn as nn
from transformers import BertModel

class QualitySharedExpert(nn.Module):
    def __init__(self, num_labels_dict):
        super().__init__()

        self.bert = BertModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        combined_size = hidden_size * 2   # CLS + Mean Pooling

        # =====================================================
        # ES / EQ 分離 projector
        # =====================================================
        self.es_projector = nn.Sequential(
            nn.Linear(combined_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.eq_projector = nn.Sequential(
            nn.Linear(combined_size, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        # =====================================================
        # ES Head：輕量，專注 evidence 有無
        # =====================================================
        self.es_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels_dict["evidence_status"])
        )

        # =====================================================
        # EQ Head：輕等深度，不要太重
        # =====================================================
        self.eq_head = nn.Sequential(

            nn.LayerNorm(512),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(256, num_labels_dict["evidence_quality"])
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        last_hidden = outputs.last_hidden_state

        # =====================================================
        # Dual Pooling: CLS + Mean Pooling
        # =====================================================
        cls_token = last_hidden[:, 0, :]

        mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()

        mean_pooled = torch.sum(last_hidden * mask, dim=1) / torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        combined = torch.cat([cls_token, mean_pooled], dim=1)

        # =====================================================
        # ES / EQ 分流
        # =====================================================
        es_feat = self.es_projector(combined)
        eq_feat = self.eq_projector(combined)

        es_logits = self.es_head(es_feat)
        eq_logits = self.eq_head(eq_feat)

        return {
            "evidence_status": es_logits,
            "evidence_quality": eq_logits
        }
  # 確保你剛才定義的 expert_label_counts 就在這格之前執行過
model_quality = QualitySharedExpert(expert_label_counts).to(device)
print("✅ 多任務模型 QualitySharedExpert 初始化成功！")

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 多任務模型 QualitySharedExpert 初始化成功！


In [ ]:
# ============================================================
# 2. Timeline Guardian: 專攻 VT (權重 15%)
# 核心：使用 pooler_output 捕捉全域時態資訊，搭配 Focal Loss
# ============================================================
class TimelineGuardian(nn.Module):
    def __init__(self, num_labels_dict):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size

        self.vt_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels_dict['verification_timeline'])
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # VT (時程) 通常由整句語氣決定（過去式/未來式），使用 pooler_output 最穩定
        pooled = outputs.pooler_output
        return {"verification_timeline": self.vt_head(pooled)}

# ============================================================
# 3. Anchor Model: 專攻 PS (權重 20%)
# 核心：基礎穩定的分類器，作為整個系統的邏輯開關 (Gate)
# ============================================================
class AnchorModel(nn.Module):
    def __init__(self, num_labels_dict):
        super().__init__()
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size

        self.ps_head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256), # 使用 LayerNorm 增加穩定性
            nn.ReLU(),
            nn.Linear(256, num_labels_dict['promise_status'])
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # PS (承諾狀態) 判斷「有沒有」，同樣依賴全域語意
        return {"promise_status": self.ps_head(outputs.pooler_output)}

# --- 初始化剩下的模型 ---
model_timeline = TimelineGuardian(expert_label_counts).to(device)
model_anchor = AnchorModel(expert_label_counts).to(device)

print("✅ 三大專家模型 (Quality, Timeline, Anchor) 全部初始化成功！")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 三大專家模型 (Quality, Timeline, Anchor) 全部初始化成功！


增加focal loss類別

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # 先算 cross entropy
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.weight,
            reduction="none"
        )

        # pt = 預測正確類別的機率
        pt = torch.exp(-ce_loss)

        # focal loss
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss



---



## Step 7: 定義訓練與評估函數

### 損失函數
我們使用 **CrossEntropyLoss**（交叉熵損失），適用於多分類問題。

對於多任務，損失 = 各任務損失的加總：
```
total_loss = loss(promise_status) + loss(timeline) + loss(evidence_status) + loss(evidence_quality)
```

### 優化器與學習率排程
- **AdamW**：Adam 的改進版，加入 weight decay 防止過擬合
- **Linear Warmup + Decay**：先線性增加學習率（warmup），再線性衰減
  - 前 10% 步數：學習率從 0 增加到 LR
  - 後 90% 步數：學習率從 LR 線性衰減到 0

### Gradient Clipping
`clip_grad_norm_(model.parameters(), 1.0)` — 防止梯度爆炸，將梯度範數裁剪到最大為 1.0

Step 1：先算 class weights（用 train_df）
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def get_class_weights(df, field, label2id):
    labels = df[field].values
    classes = list(label2id[field].keys())

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(classes),
        y=labels
    )

    return {label2id[field][cls]: w for cls, w in zip(classes, weights)}
🔹 Step 2：計算四個任務的 weights
promise_weights = get_class_weights(train_df, "promise_status", label2id)
timeline_weights = get_class_weights(train_df, "verification_timeline", label2id)
evidence_weights = get_class_weights(train_df, "evidence_status", label2id)
quality_weights = get_class_weights(train_df, "evidence_quality", label2id)
🔹 Step 3：轉成 tensor（🔥你缺這步）
import torch

promise_w = torch.tensor(list(promise_weights.values()), dtype=torch.float).to(device)
timeline_w = torch.tensor(list(timeline_weights.values()), dtype=torch.float).to(device)
evidence_w = torch.tensor(list(evidence_weights.values()), dtype=torch.float).to(device)
quality_w = torch.tensor(list(quality_weights.values()), dtype=torch.float).to(device)
🔹 Step 4：現在才能建立 criterions
import torch.nn as nn

criterions = {
    "promise_status": nn.CrossEntropyLoss(weight=promise_w),
    "verification_timeline": nn.CrossEntropyLoss(weight=timeline_w),
    "evidence_status": nn.CrossEntropyLoss(weight=evidence_w),
    "evidence_quality": nn.CrossEntropyLoss(weight=quality_w),
}

In [ ]:
df = pd.DataFrame(all_data)

In [ ]:
df["verification_timeline"] = (
    df["verification_timeline"]
    .fillna("N/A")
    .astype(str)
    .str.strip()
    .replace({
        "longer_than_5_years": "more_than_5_years",
        "nan": "N/A"
    })
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

def get_class_weights(df, field, label2id):
    labels = df[field].values
    classes = list(label2id[field].keys())

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(classes),
        y=labels
    )

    return {label2id[field][cls]: w for cls, w in zip(classes, weights)}

In [ ]:
promise_weights = get_class_weights(df, "promise_status", label2id)
timeline_weights = get_class_weights(df, "verification_timeline", label2id)
evidence_weights = get_class_weights(df, "evidence_status", label2id)
quality_weights = get_class_weights(df,"evidence_quality", label2id)

promise_status extra labels: set()
verification_timeline extra labels: set()
evidence_status extra labels: set()
evidence_quality extra labels: set()


In [ ]:
print(df["verification_timeline"].value_counts())
print(label2id["verification_timeline"])

verification_timeline
already                  718
between_2_and_5_years    498
more_than_5_years        377
N/A                      373
within_2_years            34
Name: count, dtype: int64
{'already': 0, 'within_2_years': 1, 'between_2_and_5_years': 2, 'more_than_5_years': 3, 'N/A': 4}


In [ ]:
import torch

promise_w = torch.tensor(list(promise_weights.values()), dtype=torch.float).to(device)
timeline_w = torch.tensor(list(timeline_weights.values()), dtype=torch.float).to(device)
evidence_w = torch.tensor(list(evidence_weights.values()), dtype=torch.float).to(device)
quality_w = torch.tensor(list(quality_weights.values()), dtype=torch.float).to(device)

In [ ]:
# ============================================================
# 三專家專屬損失函數設定
# ============================================================

# --- 1. Quality Expert (ES + EQ) ---
# 為了找回 ES (30%) 的巔峰分數，我們維持 CE，但你可以考慮給 ES 加一點權重
quality_criterions = {
    "evidence_status": nn.CrossEntropyLoss(weight=evidence_w),
    "evidence_quality": nn.CrossEntropyLoss(weight=quality_w)
}

# --- 2. Timeline Guardian (VT) ---
# 這是你最成功的改進點！維持 Focal Loss 來拯救 1.2% 的極稀有類別
timeline_criterions = {
    "verification_timeline": FocalLoss(gamma=2.0, weight=timeline_w)
}

# --- 3. Anchor Model (PS) ---
# 根據你的數據，PS 如果已經很穩，CE 即可；
# 若想復刻 V3 表現，可以維持 Focal Loss。
ps_class_weights = torch.tensor([1.2, 1.0]).to(device)
anchor_criterions = {
    #"promise_status": FocalLoss(gamma=2.0, weight=promise_w) # 復刻 V3 巔峰用
    #"promise_status": nn.CrossEntropyLoss(weight=promise_w) # 若追求穩定則用這個

    "promise_status": nn.CrossEntropyLoss(weight=ps_class_weights)
}

In [ ]:
# ============================================================
# 三專家專屬損失函數設定
# ============================================================

# 1. Quality Expert: ES + EQ
quality_criterions = {
    "evidence_status": nn.CrossEntropyLoss(weight=evidence_w),
    "evidence_quality": nn.CrossEntropyLoss(weight=quality_w)
}

# 2. Timeline Guardian: VT
timeline_criterions = {
    "verification_timeline": FocalLoss(
        gamma=2.0,          # 建議先用 1.5，比 2.0 穩
        weight=timeline_w
    )
}

# 3. Anchor Model: PS
ps_class_weights = torch.tensor([1.2, 1.0]).to(device)

anchor_criterions = {
    "promise_status": nn.CrossEntropyLoss(
        weight=ps_class_weights
    )
}

In [ ]:
def train_expert_epoch(model, dataloader, optimizer, scheduler, device, criterions, field_weights, alpha=4):
    """
    三專家版：專用 R-Drop 訓練函數
    """
    model.train()
    total_loss = 0
    # 獲取當前模型負責的欄位 (例如: ['evidence_status', 'evidence_quality'])
    fields = list(criterions.keys())

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # R-Drop 雙重 Forward
        logits1 = model(input_ids, attention_mask)
        logits2 = model(input_ids, attention_mask)

        loss = 0
        for field in fields:
            l1 = logits1[field]
            l2 = logits2[field]
            labels = batch["labels"][field].to(device)

            # 1. 基礎任務損失 (每個分支對應自己的 Criterion)
            ce_loss = (criterions[field](l1, labels) + criterions[field](l2, labels)) / 2

            # 2. R-Drop: 強迫 Dropout 兩次的分佈一致
            p = F.log_softmax(l1, dim=-1)
            q = F.log_softmax(l2, dim=-1)
            p_tec = F.softmax(l1, dim=-1)
            q_tec = F.softmax(l2, dim=-1)

            # 對稱 KL 散度
            kl_loss = (F.kl_div(p, q_tec, reduction='batchmean') +
                       F.kl_div(q, p_tec, reduction='batchmean')) / 2

            # 3. 結合權重 (field_weights 對應全域權重)
            # 例如 EQ 是 0.35, ES 是 0.30
            current_weight = field_weights.get(field, 1.0)
            loss += current_weight * (ce_loss + alpha * kl_loss)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)
def predict(model, dataloader, device, id2label, fields):
    """
    通用專家預測函式：
    將 model 切換至 eval 模式，並根據 fields 產出預測字典。
    """
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # 取得 logits
            logits = model(input_ids, attention_mask)

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                pred_item = {}
                for field in fields:
                    # 找到該 field 最大機率的 ID 並轉回 Label 文字
                    pred_id = logits[field][i].argmax().item()
                    pred_item[field] = id2label[field][pred_id]
                all_preds.append(pred_item)

    return all_preds

def expert_inference(models, dataloader, device, id2label):
    """
    三專家統合推論：實作 PS -> ES/EQ/VT 的邏輯門禁
    """
    for m in models.values(): m.eval()
    all_final_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # 各專家各司其職
            res_anchor = models['anchor'](input_ids, attention_mask)
            res_quality = models['quality'](input_ids, attention_mask)
            res_timeline = models['timeline'](input_ids, attention_mask)

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                # 1. 抓取預測結果
                ps_pred = id2label['promise_status'][res_anchor['promise_status'][i].argmax().item()]
                es_pred = id2label['evidence_status'][res_quality['evidence_status'][i].argmax().item()]
                eq_pred = id2label['evidence_quality'][res_quality['evidence_quality'][i].argmax().item()]
                vt_pred = id2label['verification_timeline'][res_timeline['verification_timeline'][i].argmax().item()]

                # 2. 實作邏輯門禁 (最重要！)
                # 既然是教授要看的，邏輯嚴謹性第一
                if ps_pred == "No":
                    final_item = {
                        "promise_status": "No",
                        "evidence_status": "N/A",
                        "evidence_quality": "N/A",
                        "verification_timeline": "N/A"
                    }
                else:
                    final_item = {
                        "promise_status": ps_pred,
                        "evidence_status": es_pred,
                        "evidence_quality": eq_pred,
                        "verification_timeline": vt_pred
                    }
                all_final_preds.append(final_item)

    return all_final_preds

In [ ]:
def predict(model, dataloader, device, id2label, fields):
    """
    通用 expert 預測函式
    回傳 list[dict]
    例如：
    [
        {"promise_status": "Yes"},
        {"promise_status": "No"},
        ...
    ]
    """
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            logits = model(input_ids, attention_mask)

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                pred_item = {}
                for field in fields:
                    pred_id = logits[field][i].argmax(dim=-1).item()
                    pred_item[field] = id2label[field][pred_id]
                all_preds.append(pred_item)

    return all_preds

## **混合評估函數**

In [ ]:
def evaluate_hybrid(gt_data, pred_data):
    """
    混合評估函數：加權 Macro F1

    Args:
        gt_data: Ground truth 資料列表
        pred_data: 預測結果列表（順序需與 gt_data 一致）

    Returns:
        results: 各欄位的評估結果 + 最終加權分數
    """
    assert len(gt_data) == len(pred_data), \
        f"筆數不符：gt={len(gt_data)}, pred={len(pred_data)}"

    results = {}
    weighted_score = 0.0

    for field, labels in EVAL_FIELDS.items():
        y_true = [item[field] for item in gt_data]
        y_pred = [item[field] for item in pred_data]

        # Macro F1: 每個類別分別計算 F1，再取平均（不考慮類別數量）
        macro_f1 = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
        # Micro F1: 全部類別合併計算（考慮類別數量）
        micro_f1 = f1_score(y_true, y_pred, labels=labels, average="micro", zero_division=0)
        report   = classification_report(y_true, y_pred, labels=labels, zero_division=0)

        weight = FIELD_WEIGHTS.get(field, 0)
        weighted_score += macro_f1 * weight

        results[field] = {
            "macro_f1": macro_f1,
            "micro_f1": micro_f1,
            "report": report,
            "weight": weight
        }

    results["final_weighted_score"] = weighted_score
    return results


print(" 評估函數定義完成！")
print("\n Macro F1 vs Micro F1 說明：")
print("   Macro F1: 各類別同等重要，不受樣本數影響 → 用於評估少數類別")
print("   Micro F1: 樣本數多的類別影響較大 → 整體準確率指標")

 評估函數定義完成！

 Macro F1 vs Micro F1 說明：
   Macro F1: 各類別同等重要，不受樣本數影響 → 用於評估少數類別
   Micro F1: 樣本數多的類別影響較大 → 整體準確率指標


## Step 8:5-fold 訓練



> 加入區塊引述符號



現在所有準備工作都就緒，讓我們開始訓練模型！

### 訓練流程
```
for epoch in range(EPOCHS):
    1. 訓練一個 epoch（更新模型參數）
    2. 在驗證集上預測
    3. 計算評估指標
    4. 如果是最佳模型，儲存權重
```

### 預期時間（Colab T4 GPU）
- 每個 epoch 約 3-5 分鐘
- 5 個 epoch 共約 15-25 分鐘

> **提示**：如果使用 CPU，速度會慢很多（約 10倍），建議使用 Colab GPU 執行

In [ ]:
from collections import Counter

print(
    Counter(
        x["verification_timeline"]
        for x in all_data
    )
)

print(
    sorted(
        set(
            x["verification_timeline"]
            for x in all_data
        )
    )
)

Counter({'already': 718, 'between_2_and_5_years': 498, 'more_than_5_years': 377, 'N/A': 373, 'within_2_years': 34})
['N/A', 'already', 'between_2_and_5_years', 'more_than_5_years', 'within_2_years']


In [ ]:
import json
import pandas as pd
import numpy as np
import torch
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

# =====================================================
# 1. 清理 all_data 四個欄位
# =====================================================

FIELDS = [
    "promise_status",
    "evidence_status",
    "evidence_quality",
    "verification_timeline"
]

for item in all_data:
    for field in FIELDS:

        value = item.get(field, "N/A")

        if pd.isna(value):
            value = "N/A"

        value = str(value).strip()

        if value in ["", "nan", "None"]:
            value = "N/A"

        if field == "verification_timeline":
            if value == "longer_than_5_years":
                value = "more_than_5_years"

        item[field] = value

# =====================================================
# 2. 重建 df
# =====================================================

df = pd.DataFrame(all_data)

print("資料筆數:", len(df))

for field in FIELDS:
    print("\n", field)
    print(Counter(df[field]))

# =====================================================
# 3. 任務 label 設定
# =====================================================

EXPERT_QUALITY_FIELDS = {
    "evidence_status": ["Yes", "No", "N/A"],
    "evidence_quality": ["Clear", "Not Clear", "Misleading", "N/A"],
}

EXPERT_TIMELINE_FIELDS = {
    "verification_timeline": [
        "already",
        "within_2_years",
        "between_2_and_5_years",
        "more_than_5_years",
        "N/A"
    ],
}

EXPERT_ANCHOR_FIELDS = {
    "promise_status": ["No", "Yes"],
}

ALL_EXPERT_FIELDS = {
    **EXPERT_QUALITY_FIELDS,
    **EXPERT_TIMELINE_FIELDS,
    **EXPERT_ANCHOR_FIELDS
}

expert_label2id = {
    field: {lab: i for i, lab in enumerate(labels)}
    for field, labels in ALL_EXPERT_FIELDS.items()
}

expert_id2label = {
    field: {i: lab for i, lab in enumerate(labels)}
    for field, labels in ALL_EXPERT_FIELDS.items()
}

expert_label_counts = {
    field: len(labels)
    for field, labels in ALL_EXPERT_FIELDS.items()
}

expert_num_labels = expert_label_counts

# 如果你舊 code 用 label2id / id2label，也同步
label2id = expert_label2id
id2label = expert_id2label

print("\nlabel2id:")
for field in expert_label2id:
    print(field, expert_label2id[field])

# =====================================================
# 4. 檢查是否還有非法 label
# =====================================================

for field in FIELDS:
    bad = set(df[field]) - set(expert_label2id[field].keys())
    print(f"{field} bad labels:", bad)

    assert len(bad) == 0, f"{field} 有非法 label: {bad}"

print("\n✅ labels clean")

# =====================================================
# 5. class weight function
# =====================================================

def get_class_weights(df, field, label2id):
    labels = (
        df[field]
        .fillna("N/A")
        .astype(str)
        .str.strip()
        .replace({
            "nan": "N/A",
            "None": "N/A",
            "": "N/A",
            "longer_than_5_years": "more_than_5_years"
        })
        .values
    )

    classes = list(label2id[field].keys())

    extra = set(labels) - set(classes)
    print(f"{field} extra labels:", extra)

    assert len(extra) == 0, f"{field} 有不在 mapping 裡的 label: {extra}"

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.array(classes),
        y=labels
    )

    return {
        label2id[field][cls]: float(w)
        for cls, w in zip(classes, weights)
    }

# =====================================================
# 6. 計算各欄位 weights
# =====================================================

promise_weights = get_class_weights(df, "promise_status", expert_label2id)
evidence_weights = get_class_weights(df, "evidence_status", expert_label2id)
quality_weights = get_class_weights(df, "evidence_quality", expert_label2id)
timeline_weights = get_class_weights(df, "verification_timeline", expert_label2id)

print("\npromise_weights:", promise_weights)
print("evidence_weights:", evidence_weights)
print("quality_weights:", quality_weights)
print("timeline_weights:", timeline_weights)

# dict -> tensor
promise_w = torch.tensor(
    torch.tensor([1.2, 1.0], dtype=torch.float).to(device)
).to(device)

evidence_w = torch.tensor(
    [evidence_weights[i] for i in range(expert_label_counts["evidence_status"])],
    dtype=torch.float
).to(device)

quality_w = torch.tensor(
       [0.4472, 2.2222, 5.0, 0.7634],
    dtype=torch.float
).to(device)

timeline_w = torch.tensor(
    [timeline_weights[i] for i in range(expert_label_counts["verification_timeline"])],
    dtype=torch.float
).to(device)
timeline_w = torch.clamp(
    timeline_w,
    max=5.0
)
# within_2_years 很少，避免 focal weight 爆掉
timeline_w = torch.clamp(timeline_w, max=8.0)

print("\npromise_w:", promise_w)
print("evidence_w:", evidence_w)
print("quality_w:", quality_w)
print("timeline_w:", timeline_w)

# =====================================================
# 7. criterions
# =====================================================

anchor_criterions = {
    "promise_status": torch.nn.CrossEntropyLoss(
        weight=promise_w
    )
}

quality_criterions = {
    "evidence_status": torch.nn.CrossEntropyLoss(
        weight=evidence_w
    ),
    "evidence_quality": torch.nn.CrossEntropyLoss(
        weight=quality_w
    )
}

timeline_criterions = {
    "verification_timeline": FocalLoss(
        weight=timeline_w,
        gamma=2.0
    )
}

anchor_task_weights = {
    "promise_status": 1.0
}

quality_task_weights = {
    "evidence_status": 0.3,
    "evidence_quality": 0.7
}

timeline_task_weights = {
    "verification_timeline": 1.0
}

print("\n✅ criterions 建立完成")

資料筆數: 2000

 promise_status
Counter({'Yes': 1627, 'No': 373})

 evidence_status
Counter({'Yes': 1345, 'N/A': 373, 'No': 282})

 evidence_quality
Counter({'Clear': 1118, 'N/A': 655, 'Not Clear': 225, 'Misleading': 2})

 verification_timeline
Counter({'already': 718, 'between_2_and_5_years': 498, 'more_than_5_years': 377, 'N/A': 373, 'within_2_years': 34})

label2id:
evidence_status {'Yes': 0, 'No': 1, 'N/A': 2}
evidence_quality {'Clear': 0, 'Not Clear': 1, 'Misleading': 2, 'N/A': 3}
verification_timeline {'already': 0, 'within_2_years': 1, 'between_2_and_5_years': 2, 'more_than_5_years': 3, 'N/A': 4}
promise_status {'No': 0, 'Yes': 1}
promise_status bad labels: set()
evidence_status bad labels: set()
evidence_quality bad labels: set()
verification_timeline bad labels: set()

✅ labels clean
promise_status extra labels: set()
evidence_status extra labels: set()
evidence_quality extra labels: set()
verification_timeline extra labels: set()

promise_weights: {0: 2.680965147453083, 1: 0.6146

In [ ]:
from collections import Counter
import torch





# =====================================================
# 2. 固定 VT label 順序
# =====================================================

vt_labels = [
    "already",
    "within_2_years",
    "between_2_and_5_years",
    "more_than_5_years",
    "N/A"
]

expert_label2id["verification_timeline"] = {
    label: i for i, label in enumerate(vt_labels)
}

expert_id2label["verification_timeline"] = {
    i: label for i, label in enumerate(vt_labels)
}


# =====================================================
# 3. 重算 VT class weight
# =====================================================

vt_counter = Counter(
    item["verification_timeline"]
    for item in train_data
)

print("VT Counter:")
print(vt_counter)

total = sum(vt_counter.values())
num_classes = len(vt_labels)

weights = []

for label in vt_labels:
    count = vt_counter[label]
    weight = total / (num_classes * count)
    weights.append(weight)

weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

print("Original weights:", weights)


# =====================================================
# 4. 建議：避免 within_2_years 權重爆太高
# =====================================================

weights = torch.clamp(weights, max=10.0)

print("Clamped weights:", weights)


# =====================================================
# 5. 重建 timeline_criterions
# =====================================================

timeline_criterions = {
    "verification_timeline": FocalLoss(
        weight=weights,
        gamma=2.0
    )
}

timeline_task_weights = {
    "verification_timeline": 1.0
}

print("New timeline_criterions:")
print(timeline_criterions)
print(timeline_criterions["verification_timeline"].__dict__)

NameError: name 'train_data' is not defined

In [ ]:
def normalize_labels(item):
    item = item.copy()

    # VT label 統一成官方格式
    if item.get("verification_timeline") == "longer_than_5_years":
        item["verification_timeline"] = "more_than_5_years"

    return item


all_data = [
    normalize_labels(x)
    for x in (train_data + val_data)
]

print(set(x["verification_timeline"] for x in all_data))

NameError: name 'train_data' is not defined

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

labels_ps = [
    expert_label2id["promise_status"][x["promise_status"]]
    for x in all_data
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(all_data)), labels_ps)
):

    train_fold = [all_data[i] for i in train_idx]
    val_fold = [all_data[i] for i in val_idx]

    folds.append((train_fold, val_fold))

    print(
        f"Fold {fold+1}: "
        f"train={len(train_fold)}, "
        f"val={len(val_fold)}"
    )

Fold 1: train=1600, val=400
Fold 2: train=1600, val=400
Fold 3: train=1600, val=400
Fold 4: train=1600, val=400
Fold 5: train=1600, val=400


In [ ]:
import copy
import os
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

SAVE_DIR = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v4"
os.makedirs(SAVE_DIR, exist_ok=True)

anchor_fields = list(EXPERT_ANCHOR_FIELDS.keys())
quality_fields = list(EXPERT_QUALITY_FIELDS.keys())
timeline_fields = list(EXPERT_TIMELINE_FIELDS.keys())

In [ ]:
def oversample_ps_no_once(data):
    """
    對 promise_status == No 的資料複製一次
    Yes 不動
    """
    no_data = [x for x in data if x["promise_status"] == "No"]
    return data + copy.deepcopy(no_data)

In [ ]:
train_fold, val_fold = folds[0]

loaders = build_fold_loaders(
    train_fold,
    val_fold
)

In [ ]:
##VT改成featurev2
def build_fold_loaders(train_fold, val_fold):
    # PS train 做 No oversampling 一次
    train_fold_ps_aug = oversample_ps_no_once(train_fold)

    # Anchor
    anchor_train_ds = ESGDataset(
        train_fold_ps_aug,
        tokenizer,
        expert_label2id,
        anchor_fields,
        mode="anchor"
    )

    anchor_val_ds = ESGDataset(
        val_fold,
        tokenizer,
        expert_label2id,
        anchor_fields,
        mode="anchor"
    )

    anchor_train_loader = DataLoader(
        anchor_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(anchor_fields)
    )

    anchor_val_loader = DataLoader(
        anchor_val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(anchor_fields)
    )

    # Quality
    quality_train_ds = ESGDataset(
        train_fold,
        tokenizer,
        expert_label2id,
        quality_fields,
        mode="quality"
    )

    quality_val_ds = ESGDataset(
        val_fold,
        tokenizer,
        expert_label2id,
        quality_fields,
        mode="quality"
    )

    quality_train_loader = DataLoader(
        quality_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(quality_fields)
    )

    quality_val_loader = DataLoader(
        quality_val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(quality_fields)
    )


    # Timeline：FeatureV2 + company/page context
    timeline_train_ds = ESGDataset_featurev2(
        train_fold,
        tokenizer,
        expert_label2id,
        timeline_fields,

    )

    timeline_val_ds = ESGDataset_featurev2(
        val_fold,
        tokenizer,
        expert_label2id,
        timeline_fields,

    )
    timeline_train_loader = DataLoader(
        timeline_train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    timeline_val_loader = DataLoader(
        timeline_val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(timeline_fields)
    )
    return {
        "anchor_train": anchor_train_loader,
        "anchor_val": anchor_val_loader,
        "quality_train": quality_train_loader,
        "quality_val": quality_val_loader,
        "timeline_train": timeline_train_loader,
        "timeline_val": timeline_val_loader,
    }

In [ ]:
quality_val_ds = loaders["quality_val"].dataset

print(len(quality_val_ds))

400


In [ ]:
train_fold, val_fold = folds[0]

loaders = build_fold_loaders(train_fold, val_fold)

anchor_train_loader = loaders["anchor_train"]
quality_train_loader = loaders["quality_train"]
timeline_train_loader = loaders["timeline_train"]

anchor_val_loader = loaders["anchor_val"]
quality_val_loader = loaders["quality_val"]
timeline_val_loader = loaders["timeline_val"]

In [ ]:
train_fold, val_fold = folds[0]
loaders = build_fold_loaders(train_fold, val_fold)

timeline_val_ds = loaders["timeline_val"].dataset

print(type(timeline_val_ds))
print(timeline_val_ds.build_text(timeline_val_ds.data[0])[:512])

<class '__main__.ESGDataset_featurev2'>
[ESG=S] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=3] [VAGUE_HIGH=1] [CLEAR=0] [VT_ALREADY=0] [VT_SHORT=0] [VT_MID=0] [VT_LONG=0] [RAW_EVD=None] 關注人才關鍵議題，新加坡亞洲新聞臺（CNA）特別製作專題《Taiwan’s tech industry taps female talent pool amid labour shortage》，在報導中聚焦基金會所舉辦的 Girls! TECH Action 科技女孩工作坊，探討企業如何透過教育計畫來翻轉科技產業中的性別失衡，有效地減少女性在 STEM 領域的管漏現象，並鼓勵更多國高中女生投入 STEM（科學、技術、工程與數學）領域，並充分發揮女性在科技創新的潛力！在全球科技人才短缺的挑戰下，臺灣和新加坡對此議題都表現出高度關注。兩國的科技產業正積極尋找方法挖掘女性人才的潛力，希望藉此減緩人才短缺問題，並促進產業的多元樣貌化發展。透過教育和相關計畫，致力於提升女性在 STEM 領域的參與，確保科技創新能夠從更多元的觀點中受益，幫助解決當


In [ ]:
anchor_val_ds = loaders["anchor_val"].dataset

In [ ]:
print("\n[Anchor sample]")
print(anchor_val_ds.build_text(anchor_val_ds.data[0])[:300])

print("\n[Quality sample]")
print(quality_val_ds.build_text(quality_val_ds.data[0])[:300])

print("\n[Timeline V2 sample]")
print(timeline_val_ds.build_text(timeline_val_ds.data[0])[:300])


[Anchor sample]
[ESG=S] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=3] [CLEAR=0] [TIMELINE=0/0/0] [RAW_EVD=None] 關注人才關鍵議題，新加坡亞洲新聞臺（CNA）特別製作專題《Taiwan’s tech industry taps female talent pool amid labour shortage》，在報導中聚焦基金會所舉辦的 Girls! TECH Action 科技女孩工作坊，探討企業如何透過教育計畫來翻轉科技產業中的性別失衡，有效地減少女性在 STEM 領域的管漏現象，並鼓勵更多國高中女生投入 STEM（科學、技術、

[Quality sample]
[ESG=S] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=3] [CLEAR=0] [TIMELINE=0/0/0] [RAW_EVD=None] 關注人才關鍵議題，新加坡亞洲新聞臺（CNA）特別製作專題《Taiwan’s tech industry taps female talent pool amid labour shortage》，在報導中聚焦基金會所舉辦的 Girls! TECH Action 科技女孩工作坊，探討企業如何透過教育計畫來翻轉科技產業中的性別失衡，有效地減少女性在 STEM 領域的管漏現象，並鼓勵更多國高中女生投入 STEM（科學、技術、

[Timeline V2 sample]
[ESG=S] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=3] [VAGUE_HIGH=1] [CLEAR=0] [VT_ALREADY=0] [VT_SHORT=0] [VT_MID=0] [VT_LONG=0] [RAW_EVD=None] 關注人才關鍵議題，新加坡亞洲新聞臺（CNA）特別製作專題《Taiwan’s tech industry taps female talent pool amid labour shortage》，在報導中聚焦基金會所舉辦的 Girls! TECH Action 科技女孩工作坊，探討企業如何透過教育計畫來翻轉科技產業中的性別


In [ ]:
print("Anchor criterion:")
print(anchor_criterions["promise_status"])
print(anchor_criterions["promise_status"].weight)

print("\nQuality ES criterion:")
print(quality_criterions["evidence_status"])
print(quality_criterions["evidence_status"].weight)

print("\nQuality EQ criterion:")
print(quality_criterions["evidence_quality"])
print(quality_criterions["evidence_quality"].weight)

print("\nTimeline criterion:")
print(timeline_criterions["verification_timeline"])
print(timeline_criterions["verification_timeline"].weight)

Anchor criterion:
CrossEntropyLoss()
tensor([1.2000, 1.0000], device='cuda:0')

Quality ES criterion:
CrossEntropyLoss()
tensor([0.4957, 2.3641, 1.7873], device='cuda:0')

Quality EQ criterion:
CrossEntropyLoss()
tensor([0.4472, 2.2222, 5.0000, 0.7634], device='cuda:0')

Timeline criterion:
FocalLoss()
tensor([0.5571, 5.0000, 0.8032, 1.0610, 1.0724], device='cuda:0')


In [ ]:
fold_scores = []
EPOCH = 15
for fold_id in range(5):

    print(f"\n========== Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]

    loaders = build_fold_loaders(train_fold, val_fold)

    # =====================================================
    # Anchor Expert
    # =====================================================
    model_anchor = AnchorModel(expert_label_counts).to(device)

    optimizer_anchor = AdamW(
        model_anchor.parameters(),
        lr=2e-5
    )

    scheduler_anchor = get_linear_schedule_with_warmup(
        optimizer_anchor,
        num_warmup_steps=0,
        num_training_steps=len(loaders["anchor_train"]) * EPOCHS
    )

    anchor_save_path = f"{SAVE_DIR}/best_anchor_fold{fold_id+1}.pt"

    best_anchor = run_expert_training(
        expert_name=f"Anchor_Fold{fold_id+1}",
        model=model_anchor,
        train_loader=loaders["anchor_train"],
        val_loader=loaders["anchor_val"],
        optimizer=optimizer_anchor,
        scheduler=scheduler_anchor,
        criterions=anchor_criterions,
        task_weights=anchor_task_weights,
        fields=anchor_fields,
        save_path=anchor_save_path,
        alpha=1,
        device=device
    )

    # =====================================================
    # Quality Expert
    # =====================================================
    model_quality = QualitySharedExpert(expert_label_counts).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = f"{SAVE_DIR}/best_quality_fold{fold_id+1}.pt"

    best_quality = run_expert_training(
        expert_name=f"Quality_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        alpha=1,
        device=device
    )

    # =====================================================
    # Timeline Guardian
    # =====================================================
    model_timeline = TimelineGuardian(expert_label_counts).to(device)

    optimizer_timeline = AdamW(
        model_timeline.parameters(),
        lr=2e-5
    )

    scheduler_timeline = get_linear_schedule_with_warmup(
        optimizer_timeline,
        num_warmup_steps=0,
        num_training_steps=len(loaders["timeline_train"]) * EPOCHS
    )

    timeline_save_path = f"{SAVE_DIR}/best_timeline_fold{fold_id+1}.pt"

    best_timeline = run_expert_training(
        expert_name=f"Timeline_Fold{fold_id+1}",
        model=model_timeline,
        train_loader=loaders["timeline_train"],
        val_loader=loaders["timeline_val"],
        optimizer=optimizer_timeline,
        scheduler=scheduler_timeline,
        criterions=timeline_criterions,
        task_weights=timeline_task_weights,
        fields=timeline_fields,
        save_path=timeline_save_path,
        alpha=1,
        device=device
    )

    fold_scores.append({
        "fold": fold_id + 1,
        "anchor": best_anchor,
        "quality": best_quality,
        "timeline": best_timeline
    })

    print(f"\n✅ Fold {fold_id+1} done")
    print(f"Anchor : {best_anchor:.5f}")
    print(f"Quality: {best_quality:.5f}")
    print(f"Timeline: {best_timeline:.5f}")


========== Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Anchor_Fold1！ (R-Drop alpha=1)

[Anchor_Fold1] Epoch 1/10
----------------------------------------


KeyboardInterrupt: 

In [ ]:
anchor_fold_scores = []
EPOCH = 15
for fold_id in range(5):

    print(f"\n========== Anchor Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_anchor = AnchorModel(expert_label_counts).to(device)

    optimizer_anchor = AdamW(
        model_anchor.parameters(),
        lr=2e-5
    )

    scheduler_anchor = get_linear_schedule_with_warmup(
        optimizer_anchor,
        num_warmup_steps=0,
        num_training_steps=len(loaders["anchor_train"]) * EPOCHS
    )

    anchor_save_path = f"{SAVE_DIR}/best_anchor_fold{fold_id+1}.pt"

    best_anchor = run_expert_training(
        expert_name=f"Anchor_Fold{fold_id+1}",
        model=model_anchor,
        train_loader=loaders["anchor_train"],
        val_loader=loaders["anchor_val"],
        optimizer=optimizer_anchor,
        scheduler=scheduler_anchor,
        criterions=anchor_criterions,
        task_weights=anchor_task_weights,
        fields=anchor_fields,
        save_path=anchor_save_path,
        alpha=1,
        device=device
    )

    anchor_fold_scores.append({
        "fold": fold_id + 1,
        "anchor": best_anchor
    })

    print(f"✅ Anchor Fold {fold_id+1} done | Score: {best_anchor:.5f}")

anchor_score_df = pd.DataFrame(anchor_fold_scores)
print(anchor_score_df)
print("Anchor Mean:", anchor_score_df["anchor"].mean())


========== Anchor Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Anchor_Fold1！ (R-Drop alpha=1)

[Anchor_Fold1] Epoch 1/10
----------------------------------------
  平均 Loss: 0.4656 | 加權得分: 0.15358
  -> promise_status        : Macro F1 = 0.7679
  ✅ 最佳模型已儲存！得分: 0.15358

[Anchor_Fold1] Epoch 2/10
----------------------------------------
  平均 Loss: 0.2908 | 加權得分: 0.14780
  -> promise_status        : Macro F1 = 0.7390

[Anchor_Fold1] Epoch 3/10
----------------------------------------
  平均 Loss: 0.1530 | 加權得分: 0.14519
  -> promise_status        : Macro F1 = 0.7260

[Anchor_Fold1] Epoch 4/10
----------------------------------------
  平均 Loss: 0.0523 | 加權得分: 0.15427
  -> promise_status        : Macro F1 = 0.7714
  ✅ 最佳模型已儲存！得分: 0.15427

[Anchor_Fold1] Epoch 5/10
----------------------------------------
  平均 Loss: 0.0218 | 加權得分: 0.14795
  -> promise_status        : Macro F1 = 0.7398

[Anchor_Fold1] Epoch 6/10
----------------------------------------
  平均 Loss: 0.0068 | 加權得分: 0.15558
  -> promise_status        : Macro F1 = 0.7779
  ✅ 最佳模型已儲存！得分: 0.1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Anchor_Fold4！ (R-Drop alpha=1)

[Anchor_Fold4] Epoch 1/10
----------------------------------------
  平均 Loss: 0.4205 | 加權得分: 0.16104
  -> promise_status        : Macro F1 = 0.8052
  ✅ 最佳模型已儲存！得分: 0.16104

[Anchor_Fold4] Epoch 2/10
----------------------------------------
  平均 Loss: 0.2239 | 加權得分: 0.16765
  -> promise_status        : Macro F1 = 0.8383
  ✅ 最佳模型已儲存！得分: 0.16765

[Anchor_Fold4] Epoch 3/10
----------------------------------------
  平均 Loss: 0.0880 | 加權得分: 0.17210
  -> promise_status        : Macro F1 = 0.8605
  ✅ 最佳模型已儲存！得分: 0.17210

[Anchor_Fold4] Epoch 4/10
----------------------------------------
  平均 Loss: 0.0581 | 加權得分: 0.15499
  -> promise_status        : Macro F1 = 0.7749

[Anchor_Fold4] Epoch 5/10
----------------------------------------
  平均 Loss: 0.0340 | 加權得分: 0.15755
  -> promise_status        : Macro F1 = 0.7878

[Anchor_Fold4] Epoch 6/10
----------------------------------------
  平均 Loss: 0.0190 | 加權得分: 0.16623
  -> promise_status        : Macro F1 = 0.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Anchor_Fold5！ (R-Drop alpha=1)

[Anchor_Fold5] Epoch 1/10
----------------------------------------
  平均 Loss: 0.4903 | 加權得分: 0.16163
  -> promise_status        : Macro F1 = 0.8082
  ✅ 最佳模型已儲存！得分: 0.16163

[Anchor_Fold5] Epoch 2/10
----------------------------------------
  平均 Loss: 0.3178 | 加權得分: 0.15860
  -> promise_status        : Macro F1 = 0.7930

[Anchor_Fold5] Epoch 3/10
----------------------------------------
  平均 Loss: 0.1316 | 加權得分: 0.14498
  -> promise_status        : Macro F1 = 0.7249

[Anchor_Fold5] Epoch 4/10
----------------------------------------
  平均 Loss: 0.0696 | 加權得分: 0.16026
  -> promise_status        : Macro F1 = 0.8013

[Anchor_Fold5] Epoch 5/10
----------------------------------------
  平均 Loss: 0.0222 | 加權得分: 0.16346
  -> promise_status        : Macro F1 = 0.8173
  ✅ 最佳模型已儲存！得分: 0.16346

[Anchor_Fold5] Epoch 6/10
----------------------------------------
  平均 Loss: 0.0184 | 加權得分: 0.15855
  -> promise_status        : Macro F1 = 0.7928

[Anchor_Fold5] Epo

In [ ]:
print("evidence_status classes:", expert_label_counts["evidence_status"])
print("evidence_quality classes:", expert_label_counts["evidence_quality"])

print("evidence_w shape:", evidence_w.shape)
print("quality_w shape:", quality_w.shape)

print("ES criterion weight:", quality_criterions["evidence_status"].weight.shape)
print("EQ criterion weight:", quality_criterions["evidence_quality"].weight.shape)

evidence_status classes: 3
evidence_quality classes: 4
evidence_w shape: torch.Size([3])
quality_w shape: torch.Size([4])
ES criterion weight: torch.Size([3])
EQ criterion weight: torch.Size([4])


In [ ]:
quality_fold_scores = []
EPOCHS=12
for fold_id in range(5):

    print(f"\n========== Quality Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_quality = QualitySharedExpert(expert_label_counts).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = f"{SAVE_DIR}/best_quality_fold{fold_id+1}.pt"

    best_quality = run_expert_training(
        expert_name=f"Quality_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        alpha=2,
        device=device
    )

    quality_fold_scores.append({
        "fold": fold_id + 1,
        "quality": best_quality
    })

    print(f"✅ Quality Fold {fold_id+1} done | Score: {best_quality:.5f}")

quality_score_df = pd.DataFrame(quality_fold_scores)
print(quality_score_df)
print("Quality Mean:", quality_score_df["quality"].mean())


========== Quality Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_Fold1！ (R-Drop alpha=2)

[Quality_Fold1] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0303 | 加權得分: 0.32472
  -> evidence_status       : Macro F1 = 0.6020
  -> evidence_quality      : Macro F1 = 0.4118
  ✅ 最佳模型已儲存！得分: 0.32472

[Quality_Fold1] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8157 | 加權得分: 0.35709
  -> evidence_status       : Macro F1 = 0.6539
  -> evidence_quality      : Macro F1 = 0.4598
  ✅ 最佳模型已儲存！得分: 0.35709

[Quality_Fold1] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6227 | 加權得分: 0.35075
  -> evidence_status       : Macro F1 = 0.6281
  -> evidence_quality      : Macro F1 = 0.4638

[Quality_Fold1] Epoch 4/12
----------------------------------------
  平均 Loss: 0.4588 | 加權得分: 0.34573
  -> evidence_status       : Macro F1 = 0.6447
  -> evidence_quality      : Macro F1 = 0.4352

[Quality_Fold1] Epoch 5/12
----------------------------------------
  平均 Loss: 0.2797 | 加權得分: 0.34470
  -> evidence_status 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_Fold4！ (R-Drop alpha=2)

[Quality_Fold4] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0285 | 加權得分: 0.31774
  -> evidence_status       : Macro F1 = 0.5904
  -> evidence_quality      : Macro F1 = 0.4018
  ✅ 最佳模型已儲存！得分: 0.31774

[Quality_Fold4] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8244 | 加權得分: 0.31716
  -> evidence_status       : Macro F1 = 0.5970
  -> evidence_quality      : Macro F1 = 0.3944

[Quality_Fold4] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6368 | 加權得分: 0.36672
  -> evidence_status       : Macro F1 = 0.6973
  -> evidence_quality      : Macro F1 = 0.4501
  ✅ 最佳模型已儲存！得分: 0.36672

[Quality_Fold4] Epoch 4/12
----------------------------------------
  平均 Loss: 0.4530 | 加權得分: 0.35649
  -> evidence_status       : Macro F1 = 0.6470
  -> evidence_quality      : Macro F1 = 0.4640

[Quality_Fold4] Epoch 5/12
----------------------------------------
  平均 Loss: 0.2980 | 加權得分: 0.36799
  -> evidence_status 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_Fold5！ (R-Drop alpha=2)

[Quality_Fold5] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0327 | 加權得分: 0.31279
  -> evidence_status       : Macro F1 = 0.6130
  -> evidence_quality      : Macro F1 = 0.3683
  ✅ 最佳模型已儲存！得分: 0.31279

[Quality_Fold5] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8278 | 加權得分: 0.35801
  -> evidence_status       : Macro F1 = 0.6851
  -> evidence_quality      : Macro F1 = 0.4357
  ✅ 最佳模型已儲存！得分: 0.35801

[Quality_Fold5] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6719 | 加權得分: 0.34709
  -> evidence_status       : Macro F1 = 0.6427
  -> evidence_quality      : Macro F1 = 0.4408

[Quality_Fold5] Epoch 4/12
----------------------------------------
  平均 Loss: 0.4998 | 加權得分: 0.35737
  -> evidence_status       : Macro F1 = 0.6646
  -> evidence_quality      : Macro F1 = 0.4514

[Quality_Fold5] Epoch 5/12
----------------------------------------
  平均 Loss: 0.3568 | 加權得分: 0.33829
  -> evidence_status 

In [ ]:
import inspect

print(inspect.signature(run_expert_training))
print(inspect.getsource(run_expert_training))

(expert_name, model, train_loader, val_loader, optimizer, scheduler, criterions, task_weights, fields, save_path, device, alpha=4)
def run_expert_training(
    expert_name, model,
    train_loader, val_loader,
    optimizer, scheduler,
    criterions, task_weights,
    fields, save_path,
    device,
    alpha=4
):
    print(f"\n🚀 開始訓練 {expert_name}！ (R-Drop alpha={alpha})")
    print("=" * 60)

    best_score = 0.0

    for epoch in range(EPOCHS):
        print(f"\n[{expert_name}] Epoch {epoch+1}/{EPOCHS}")
        print("-" * 40)

        # 1. 訓練
        avg_loss = train_expert_epoch(
            model, train_loader, optimizer, scheduler, device,
            criterions, task_weights, alpha=alpha
        )

        # 2. 預測
        val_preds_short = predict(
            model, val_loader, device,
            expert_id2label, fields
        )

        # ⭐ 用 dataset.data 確保順序一致
        val_source = val_loader.dataset.data

        val_gt_full = []
        val_preds_full = []

        for 

In [ ]:
timeline_fold_scores = []

for fold_id in range(5):

    print(f"\n========== Timeline Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_timeline = TimelineGuardian(expert_label_counts).to(device)

    optimizer_timeline = AdamW(
        model_timeline.parameters(),
        lr=2e-5
    )

    scheduler_timeline = get_linear_schedule_with_warmup(
        optimizer_timeline,
        num_warmup_steps=0,
        num_training_steps=len(loaders["timeline_train"]) * EPOCHS
    )

    timeline_save_path = f"{SAVE_DIR}/best_timeline_fold_new{fold_id+1}.pt"

    best_timeline = run_expert_training(
        expert_name=f"Timeline_Fold{fold_id+1}",
        model=model_timeline,
        train_loader=loaders["timeline_train"],
        val_loader=loaders["timeline_val"],
        optimizer=optimizer_timeline,
        scheduler=scheduler_timeline,
        criterions=timeline_criterions,
        task_weights=timeline_task_weights,
        fields=timeline_fields,
        save_path=timeline_save_path,
        alpha=1,
        device=device
    )

    timeline_fold_scores.append({
        "fold": fold_id + 1,
        "timeline": best_timeline
    })

    print(f"✅ Timeline Fold {fold_id+1} done | Score: {best_timeline:.5f}")

timeline_score_df = pd.DataFrame(timeline_fold_scores)
print(timeline_score_df)
print("Timeline Mean:", timeline_score_df["timeline"].mean())


========== Timeline Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold1！ (R-Drop alpha=1)

[Timeline_Fold1] Epoch 1/12
----------------------------------------
  平均 Loss: 0.7424 | 加權得分: 0.06108
  -> verification_timeline : Macro F1 = 0.4072
  ✅ 最佳模型已儲存！得分: 0.06108

[Timeline_Fold1] Epoch 2/12
----------------------------------------
  平均 Loss: 0.5272 | 加權得分: 0.08359
  -> verification_timeline : Macro F1 = 0.5573
  ✅ 最佳模型已儲存！得分: 0.08359

[Timeline_Fold1] Epoch 3/12
----------------------------------------
  平均 Loss: 0.3609 | 加權得分: 0.08037
  -> verification_timeline : Macro F1 = 0.5358

[Timeline_Fold1] Epoch 4/12
----------------------------------------
  平均 Loss: 0.2112 | 加權得分: 0.08418
  -> verification_timeline : Macro F1 = 0.5612
  ✅ 最佳模型已儲存！得分: 0.08418

[Timeline_Fold1] Epoch 5/12
----------------------------------------
  平均 Loss: 0.1333 | 加權得分: 0.08903
  -> verification_timeline : Macro F1 = 0.5936
  ✅ 最佳模型已儲存！得分: 0.08903

[Timeline_Fold1] Epoch 6/12
----------------------------------------
  平均 Loss: 0.0831 | 加權得分: 0.09008
  ->

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold4！ (R-Drop alpha=1)

[Timeline_Fold4] Epoch 1/12
----------------------------------------
  平均 Loss: 0.7726 | 加權得分: 0.05066
  -> verification_timeline : Macro F1 = 0.3377
  ✅ 最佳模型已儲存！得分: 0.05066

[Timeline_Fold4] Epoch 2/12
----------------------------------------
  平均 Loss: 0.5711 | 加權得分: 0.06123
  -> verification_timeline : Macro F1 = 0.4082
  ✅ 最佳模型已儲存！得分: 0.06123

[Timeline_Fold4] Epoch 3/12
----------------------------------------
  平均 Loss: 0.3751 | 加權得分: 0.07429
  -> verification_timeline : Macro F1 = 0.4953
  ✅ 最佳模型已儲存！得分: 0.07429

[Timeline_Fold4] Epoch 4/12
----------------------------------------
  平均 Loss: 0.2638 | 加權得分: 0.07213
  -> verification_timeline : Macro F1 = 0.4809

[Timeline_Fold4] Epoch 5/12
----------------------------------------
  平均 Loss: 0.1831 | 加權得分: 0.08180
  -> verification_timeline : Macro F1 = 0.5454
  ✅ 最佳模型已儲存！得分: 0.08180

[Timeline_Fold4] Epoch 6/12
----------------------------------------
  平均 Loss: 0.1172 | 加權得分: 0.08882
  ->

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold5！ (R-Drop alpha=1)

[Timeline_Fold5] Epoch 1/12
----------------------------------------
  平均 Loss: 0.7083 | 加權得分: 0.05529
  -> verification_timeline : Macro F1 = 0.3686
  ✅ 最佳模型已儲存！得分: 0.05529

[Timeline_Fold5] Epoch 2/12
----------------------------------------
  平均 Loss: 0.5369 | 加權得分: 0.07737
  -> verification_timeline : Macro F1 = 0.5158
  ✅ 最佳模型已儲存！得分: 0.07737

[Timeline_Fold5] Epoch 3/12
----------------------------------------
  平均 Loss: 0.3934 | 加權得分: 0.08777
  -> verification_timeline : Macro F1 = 0.5851
  ✅ 最佳模型已儲存！得分: 0.08777

[Timeline_Fold5] Epoch 4/12
----------------------------------------
  平均 Loss: 0.2646 | 加權得分: 0.09181
  -> verification_timeline : Macro F1 = 0.6121
  ✅ 最佳模型已儲存！得分: 0.09181

[Timeline_Fold5] Epoch 5/12
----------------------------------------
  平均 Loss: 0.1730 | 加權得分: 0.09111
  -> verification_timeline : Macro F1 = 0.6074

[Timeline_Fold5] Epoch 6/12
----------------------------------------
  平均 Loss: 0.1185 | 加權得分: 0.08983
  ->

In [ ]:
train_fold, val_fold = folds[0]

loaders = build_fold_loaders(train_fold, val_fold)

print(type(loaders["timeline_train"].dataset))
print(type(loaders["timeline_val"].dataset))



<class '__main__.ESGDataset'>
<class '__main__.ESGDataset'>


In [ ]:
import gc
import torch

# 刪掉大型模型清單
for name in [
    "anchor_models",
    "quality_models",
    "quality_models_fgm",
    "quality_models_nofgm",
    "timeline_models",
    "timeline_models_v2",
    "timeline_models_testsafe",
    "model_anchor",
    "model_quality",
    "model_timeline"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 2            |        cudaMalloc retries: 3         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4033 MiB |  21404 MiB |  29068 GiB |  29064 GiB |
|       from large pool |   4019 MiB |  21351 MiB |  29054 GiB |  29050 GiB |
|       from small pool |     14 MiB |     53 MiB |     13 GiB |     13 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4033 MiB |  21404 MiB |  29068 GiB |  29064 GiB |
|       from large pool |   4019 MiB |  21351 MiB |  29054 GiB |

In [ ]:
timeline_fold_scores = []

for fold_id in range(5):

    print(f"\n========== Timeline Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_timeline = TimelineGuardian(expert_label_counts).to(device)

    optimizer_timeline = AdamW(
        model_timeline.parameters(),
        lr=2e-5
    )

    scheduler_timeline = get_linear_schedule_with_warmup(
        optimizer_timeline,
        num_warmup_steps=0,
        num_training_steps=len(loaders["timeline_train"]) * EPOCHS
    )

    timeline_save_path = f"{SAVE_DIR}/best_timeline_fold_new2{fold_id+1}.pt"

    best_timeline = run_expert_training(
        expert_name=f"Timeline_Fold{fold_id+1}",
        model=model_timeline,
        train_loader=loaders["timeline_train"],
        val_loader=loaders["timeline_val"],
        optimizer=optimizer_timeline,
        scheduler=scheduler_timeline,
        criterions=timeline_criterions,
        task_weights=timeline_task_weights,
        fields=timeline_fields,
        save_path=timeline_save_path,
        alpha=1,
        device=device
    )

    timeline_fold_scores.append({
        "fold": fold_id + 1,
        "timeline": best_timeline
    })

    print(f"✅ Timeline Fold {fold_id+1} done | Score: {best_timeline:.5f}")

timeline_score_df = pd.DataFrame(timeline_fold_scores)
print(timeline_score_df)
print("Timeline Mean:", timeline_score_df["timeline"].mean())


========== Timeline Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold1！ (R-Drop alpha=1)

[Timeline_Fold1] Epoch 1/10
----------------------------------------
  平均 Loss: 1.1132 | 加權得分: 0.05041
  -> verification_timeline : Macro F1 = 0.3361
  ✅ 最佳模型已儲存！得分: 0.05041

[Timeline_Fold1] Epoch 2/10
----------------------------------------
  平均 Loss: 0.9555 | 加權得分: 0.06371
  -> verification_timeline : Macro F1 = 0.4247
  ✅ 最佳模型已儲存！得分: 0.06371

[Timeline_Fold1] Epoch 3/10
----------------------------------------
  平均 Loss: 0.7020 | 加權得分: 0.05799
  -> verification_timeline : Macro F1 = 0.3866

[Timeline_Fold1] Epoch 4/10
----------------------------------------
  平均 Loss: 0.5947 | 加權得分: 0.06889
  -> verification_timeline : Macro F1 = 0.4593
  ✅ 最佳模型已儲存！得分: 0.06889

[Timeline_Fold1] Epoch 5/10
----------------------------------------
  平均 Loss: 0.4528 | 加權得分: 0.08692
  -> verification_timeline : Macro F1 = 0.5794
  ✅ 最佳模型已儲存！得分: 0.08692

[Timeline_Fold1] Epoch 6/10
----------------------------------------
  平均 Loss: 0.2985 | 加權得分: 0.09104
  ->

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold4！ (R-Drop alpha=1)

[Timeline_Fold4] Epoch 1/10
----------------------------------------
  平均 Loss: 1.1413 | 加權得分: 0.02986
  -> verification_timeline : Macro F1 = 0.1991
  ✅ 最佳模型已儲存！得分: 0.02986

[Timeline_Fold4] Epoch 2/10
----------------------------------------
  平均 Loss: 0.9527 | 加權得分: 0.06105
  -> verification_timeline : Macro F1 = 0.4070
  ✅ 最佳模型已儲存！得分: 0.06105

[Timeline_Fold4] Epoch 3/10
----------------------------------------
  平均 Loss: 0.7534 | 加權得分: 0.07096
  -> verification_timeline : Macro F1 = 0.4731
  ✅ 最佳模型已儲存！得分: 0.07096

[Timeline_Fold4] Epoch 4/10
----------------------------------------
  平均 Loss: 0.5632 | 加權得分: 0.07200
  -> verification_timeline : Macro F1 = 0.4800
  ✅ 最佳模型已儲存！得分: 0.07200

[Timeline_Fold4] Epoch 5/10
----------------------------------------
  平均 Loss: 0.4363 | 加權得分: 0.09591
  -> verification_timeline : Macro F1 = 0.6394
  ✅ 最佳模型已儲存！得分: 0.09591

[Timeline_Fold4] Epoch 6/10
----------------------------------------
  平均 Loss: 0.2

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_Fold5！ (R-Drop alpha=1)

[Timeline_Fold5] Epoch 1/10
----------------------------------------
  平均 Loss: 1.0458 | 加權得分: 0.03347
  -> verification_timeline : Macro F1 = 0.2231
  ✅ 最佳模型已儲存！得分: 0.03347

[Timeline_Fold5] Epoch 2/10
----------------------------------------
  平均 Loss: 1.0196 | 加權得分: 0.05110
  -> verification_timeline : Macro F1 = 0.3407
  ✅ 最佳模型已儲存！得分: 0.05110

[Timeline_Fold5] Epoch 3/10
----------------------------------------
  平均 Loss: 0.7101 | 加權得分: 0.06336
  -> verification_timeline : Macro F1 = 0.4224
  ✅ 最佳模型已儲存！得分: 0.06336

[Timeline_Fold5] Epoch 4/10
----------------------------------------
  平均 Loss: 0.4885 | 加權得分: 0.06935
  -> verification_timeline : Macro F1 = 0.4624
  ✅ 最佳模型已儲存！得分: 0.06935

[Timeline_Fold5] Epoch 5/10
----------------------------------------
  平均 Loss: 0.2477 | 加權得分: 0.06972
  -> verification_timeline : Macro F1 = 0.4648
  ✅ 最佳模型已儲存！得分: 0.06972

[Timeline_Fold5] Epoch 6/10
----------------------------------------
  平均 Loss: 0.1

##testsafe版本timeline

---



In [ ]:
import re
import os
import json
import torch
import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

# =====================================================
# 2. TestSafe Timeline Feature
# =====================================================

def extract_raw_evidence(text):
    patterns = (
        r'(\d+(\.\d+)?%|'
        r'ISO\s*\d+|'
        r'\d{4}年|'
        r'\d+\s*(?:公噸|公斤|度|kWh|MW|億元|萬元|天|週|年))'
    )

    matches = re.findall(patterns, str(text))

    found = [
        m[0] if isinstance(m, tuple) else m
        for m in matches
    ]

    return "/".join(found[:5]) if found else "None"


def extract_target_year(text):
    years = re.findall(r"20\d{2}", str(text))

    if len(years) == 0:
        return -1

    years = [int(y) for y in years]

    future_years = [y for y in years if y >= 2024]

    if len(future_years) == 0:
        return max(years)

    return min(future_years)


def build_timeline_prefix_testsafe(item):
    text = str(item.get("data", ""))

    target_year = extract_target_year(text)

    if target_year == -1:
        year_diff = -1
        year_bucket = "UNK"
    else:
        year_diff = target_year - 2024

        if year_diff <= 0:
            year_bucket = "ALREADY"
        elif year_diff <= 2:
            year_bucket = "SHORT"
        elif year_diff <= 5:
            year_bucket = "MID"
        else:
            year_bucket = "LONG"

    completed_words = [
        "已完成", "已達成", "已取得", "已建置",
        "已導入", "通過", "完成", "截至"
    ]

    annual_words = [
        "每年", "年度", "逐年", "定期", "每季",
        "每月", "持續追蹤", "持續檢討"
    ]

    continue_words = [
        "持續", "長年", "逐步", "規劃",
        "推動", "落實", "深化", "精進"
    ]

    short_words = [
        "短期", "近期", "兩年內", "2025", "2026"
    ]

    mid_words = [
        "中期", "五年內", "2027", "2028", "2029", "2030"
    ]

    long_words = [
        "長期", "長遠", "永續經營", "淨零",
        "2050", "2040"
    ]

    has_completed = int(any(w in text for w in completed_words))
    has_annual = int(any(w in text for w in annual_words))
    has_continue = int(any(w in text for w in continue_words))
    has_short = int(any(w in text for w in short_words))
    has_mid = int(any(w in text for w in mid_words))
    has_long = int(any(w in text for w in long_words))

    has_number = int(bool(re.search(r"\d", text)))
    has_year = int(bool(re.search(r"20\d{2}", text)))
    has_percent = int(bool(re.search(r"%|％", text)))

    raw_time = extract_raw_evidence(text)

    company = item.get("company", "UNK")
    ticker = item.get("ticker", "UNK")
    page_number = item.get("page_number", "UNK")

    return (
        f"[COMPANY={company}] "
        f"[TICKER={ticker}] "
        f"[PAGE={page_number}] "
        f"[NUM={has_number}] "
        f"[YEAR={has_year}] "
        f"[PERCENT={has_percent}] "
        f"[TARGET_YEAR={target_year}] "
        f"[YEAR_DIFF={year_diff}] "
        f"[YEAR_BUCKET={year_bucket}] "
        f"[HAS_COMPLETED={has_completed}] "
        f"[HAS_ANNUAL={has_annual}] "
        f"[HAS_CONTINUE={has_continue}] "
        f"[HAS_SHORT={has_short}] "
        f"[HAS_MID={has_mid}] "
        f"[HAS_LONG={has_long}] "
        f"[TIME={raw_time}] "
    )
# =====================================================
# 3. TestSafe Timeline Dataset
# =====================================================

class ESGTimelineDataset_TestSafe(Dataset):

    def __init__(self, data, tokenizer, label2id, fields):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.fields = fields

    def __len__(self):
        return len(self.data)

    def build_text(self, item):
        prefix = build_timeline_prefix_testsafe(item)
        text = str(item.get("data", ""))

        return prefix + "[TEXT] " + text

    def __getitem__(self, idx):
        item = self.data[idx]
        text = self.build_text(item)

        encoding = tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}

        for field in self.fields:
            mapping = self.label2id[field]

            if field in item:
                label = item[field]

                if field == "verification_timeline" and label == "longer_than_5_years":
                    label = "more_than_5_years"

                labels[field] = mapping[label]
            else:
                labels[field] = -1

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }
# =====================================================
# 4. 重建 FocalLoss weight
# =====================================================

vt_counter = Counter(x["verification_timeline"] for x in all_data)

total = sum(vt_counter.values())
num_classes = len(vt_labels)

weights = []

for label in vt_labels:
    count = vt_counter[label]
    weight = total / (num_classes * count)
    weights.append(weight)

weights = torch.tensor(weights, dtype=torch.float).to(device)
weights = torch.clamp(weights, max=8.0)

print("VT Counter:", vt_counter)
print("VT weights:", weights)

timeline_criterions = {
    "verification_timeline": FocalLoss(
        weight=weights,
        gamma=2.0
    )
}

timeline_task_weights = {
    "verification_timeline": 1.0
}
# =====================================================
# 5. Stratified 5-fold by VT
# =====================================================

vt_y = [
    expert_label2id["verification_timeline"][x["verification_timeline"]]
    for x in all_data
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

timeline_folds = []

for fold_id, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(all_data)), vt_y)
):
    train_fold = [all_data[i] for i in train_idx]
    val_fold = [all_data[i] for i in val_idx]

    timeline_folds.append((train_fold, val_fold))

    print(f"Fold {fold_id+1}")
    print("Train:", Counter(x["verification_timeline"] for x in train_fold))
    print("Val  :", Counter(x["verification_timeline"] for x in val_fold))
# =====================================================
# 6. 建立 loader
# =====================================================

def build_timeline_testsafe_loaders(train_fold, val_fold):
    train_ds = ESGTimelineDataset_TestSafe(
        train_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    val_ds = ESGTimelineDataset_TestSafe(
        val_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    return train_loader, val_loader

VT Counter: Counter({'already': 718, 'between_2_and_5_years': 498, 'more_than_5_years': 377, 'N/A': 373, 'within_2_years': 34})
VT weights: tensor([0.5571, 8.0000, 0.8032, 1.0610, 1.0724], device='cuda:0')
Fold 1
Train: Counter({'already': 574, 'between_2_and_5_years': 398, 'more_than_5_years': 301, 'N/A': 299, 'within_2_years': 28})
Val  : Counter({'already': 144, 'between_2_and_5_years': 100, 'more_than_5_years': 76, 'N/A': 74, 'within_2_years': 6})
Fold 2
Train: Counter({'already': 574, 'between_2_and_5_years': 399, 'more_than_5_years': 302, 'N/A': 298, 'within_2_years': 27})
Val  : Counter({'already': 144, 'between_2_and_5_years': 99, 'more_than_5_years': 75, 'N/A': 75, 'within_2_years': 7})
Fold 3
Train: Counter({'already': 574, 'between_2_and_5_years': 399, 'more_than_5_years': 302, 'N/A': 298, 'within_2_years': 27})
Val  : Counter({'already': 144, 'between_2_and_5_years': 99, 'more_than_5_years': 75, 'N/A': 75, 'within_2_years': 7})
Fold 4
Train: Counter({'already': 575, 'betwee

In [ ]:

# =====================================================
# 7. 訓練 Timeline TestSafe 5-fold
# =====================================================

SAVE_DIR_VT_TESTSAFE = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v4"
os.makedirs(SAVE_DIR_VT_TESTSAFE, exist_ok=True)

timeline_testsafe_scores = []

for fold_id in range(5):

    print(f"\n========== Timeline TestSafe Fold {fold_id+1}/5 ==========")

    train_fold, val_fold = timeline_folds[fold_id]

    train_loader, val_loader = build_timeline_testsafe_loaders(
        train_fold,
        val_fold
    )

    if fold_id == 0:
        ds = train_loader.dataset
        print(ds.build_text(ds.data[0])[:500])

    model_timeline = TimelineGuardian(
        expert_label_counts
    ).to(device)

    optimizer_timeline = AdamW(
        model_timeline.parameters(),
        lr=2e-5
    )

    scheduler_timeline = get_linear_schedule_with_warmup(
        optimizer_timeline,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * EPOCHS
    )

    save_path = f"{SAVE_DIR_VT_TESTSAFE}/best_timeline_testsafe_fold{fold_id+1}.pt"

    best_vt = run_expert_training(
        expert_name=f"Timeline_TestSafe_Fold{fold_id+1}",
        model=model_timeline,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer_timeline,
        scheduler=scheduler_timeline,
        criterions=timeline_criterions,
        task_weights=timeline_task_weights,
        fields=timeline_fields,
        save_path=save_path,
        alpha=1,
        device=device
    )

    timeline_testsafe_scores.append({
        "fold": fold_id + 1,
        "timeline_testsafe": best_vt
    })

    print(f"✅ Fold {fold_id+1} done | VT={best_vt:.5f}")

timeline_testsafe_score_df = pd.DataFrame(timeline_testsafe_scores)
print(timeline_testsafe_score_df)
print("Timeline TestSafe Mean:", timeline_testsafe_score_df["timeline_testsafe"].mean())

VT 分布:
Counter({'already': 366, 'between_2_and_5_years': 238, 'more_than_5_years': 197, 'N/A': 186, 'within_2_years': 13})
{'already': 0, 'within_2_years': 1, 'between_2_and_5_years': 2, 'more_than_5_years': 3, 'N/A': 4}
VT Counter: Counter({'already': 366, 'between_2_and_5_years': 238, 'more_than_5_years': 197, 'N/A': 186, 'within_2_years': 13})
VT weights: tensor([0.5464, 8.0000, 0.8403, 1.0152, 1.0753], device='cuda:0')
Fold 1
Train: Counter({'already': 292, 'between_2_and_5_years': 191, 'more_than_5_years': 157, 'N/A': 149, 'within_2_years': 11})
Val  : Counter({'already': 74, 'between_2_and_5_years': 47, 'more_than_5_years': 40, 'N/A': 37, 'within_2_years': 2})
Fold 2
Train: Counter({'already': 293, 'between_2_and_5_years': 190, 'more_than_5_years': 157, 'N/A': 149, 'within_2_years': 11})
Val  : Counter({'already': 73, 'between_2_and_5_years': 48, 'more_than_5_years': 40, 'N/A': 37, 'within_2_years': 2})
Fold 3
Train: Counter({'already': 293, 'between_2_and_5_years': 190, 'more_th

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_TestSafe_Fold1！ (R-Drop alpha=1)

[Timeline_TestSafe_Fold1] Epoch 1/10
----------------------------------------


KeyboardInterrupt: 

In [ ]:
import re
import os
import json
import torch
import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

# =====================================================
# 2. TestSafe Timeline Feature
# =====================================================

def extract_raw_evidence(text):
    patterns = (
        r'(\d+(\.\d+)?%|'
        r'ISO\s*\d+|'
        r'\d{4}年|'
        r'\d+\s*(?:公噸|公斤|度|kWh|MW|億元|萬元|天|週|年))'
    )

    matches = re.findall(patterns, str(text))

    found = [
        m[0] if isinstance(m, tuple) else m
        for m in matches
    ]

    return "/".join(found[:5]) if found else "None"


def extract_target_year(text):
    years = re.findall(r"20\d{2}", str(text))

    if len(years) == 0:
        return -1

    years = [int(y) for y in years]

    future_years = [y for y in years if y >= 2024]

    if len(future_years) == 0:
        return max(years)

    return min(future_years)


def build_timeline_prefix_testsafe(item):
    text = str(item.get("data", ""))

    target_year = extract_target_year(text)

    if target_year == -1:
        year_diff = -1
        year_bucket = "UNK"
    else:
        year_diff = target_year - 2024

        if year_diff <= 0:
            year_bucket = "ALREADY"
        elif year_diff <= 2:
            year_bucket = "SHORT"
        elif year_diff <= 5:
            year_bucket = "MID"
        else:
            year_bucket = "LONG"

    completed_words = [
        "已完成", "已達成", "已取得", "已建置",
        "已導入", "通過", "完成", "截至"
    ]

    annual_words = [
        "每年", "年度", "逐年", "定期", "每季",
        "每月", "持續追蹤", "持續檢討"
    ]

    continue_words = [
        "持續", "長年", "逐步", "規劃",
        "推動", "落實", "深化", "精進"
    ]

    short_words = [
        "短期", "近期", "兩年內", "2025", "2026"
    ]

    mid_words = [
        "中期", "五年內", "2027", "2028", "2029", "2030"
    ]

    long_words = [
        "長期", "長遠", "永續經營", "淨零",
        "2050", "2040"
    ]

    has_completed = int(any(w in text for w in completed_words))
    has_annual = int(any(w in text for w in annual_words))
    has_continue = int(any(w in text for w in continue_words))
    has_short = int(any(w in text for w in short_words))
    has_mid = int(any(w in text for w in mid_words))
    has_long = int(any(w in text for w in long_words))

    has_number = int(bool(re.search(r"\d", text)))
    has_year = int(bool(re.search(r"20\d{2}", text)))
    has_percent = int(bool(re.search(r"%|％", text)))

    raw_time = extract_raw_evidence(text)

    company = item.get("company", "UNK")
    ticker = item.get("ticker", "UNK")
    page_number = item.get("page_number", "UNK")

    return (
        f"[COMPANY={company}] "
        f"[TICKER={ticker}] "
        f"[PAGE={page_number}] "
        f"[NUM={has_number}] "
        f"[YEAR={has_year}] "
        f"[PERCENT={has_percent}] "
        f"[TARGET_YEAR={target_year}] "
        f"[YEAR_DIFF={year_diff}] "
        f"[YEAR_BUCKET={year_bucket}] "
        f"[HAS_COMPLETED={has_completed}] "
        f"[HAS_ANNUAL={has_annual}] "
        f"[HAS_CONTINUE={has_continue}] "
        f"[HAS_SHORT={has_short}] "
        f"[HAS_MID={has_mid}] "
        f"[HAS_LONG={has_long}] "
        f"[TIME={raw_time}] "
    )
# =====================================================
# 3. TestSafe Timeline Dataset
# =====================================================

class ESGTimelineDataset_TestSafe(Dataset):

    def __init__(self, data, tokenizer, label2id, fields):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.fields = fields

    def __len__(self):
        return len(self.data)

    def build_text(self, item):
        prefix = build_timeline_prefix_testsafe(item)
        text = str(item.get("data", ""))

        return prefix + "[TEXT] " + text

    def __getitem__(self, idx):
        item = self.data[idx]
        text = self.build_text(item)

        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}

        for field in self.fields:
            mapping = self.label2id[field]

            if field in item:
                label = item[field]

                if field == "verification_timeline" and label == "longer_than_5_years":
                    label = "more_than_5_years"

                labels[field] = mapping[label]
            else:
                labels[field] = -1

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }
# =====================================================
# 4. 重建 FocalLoss weight
# =====================================================

vt_counter = Counter(x["verification_timeline"] for x in all_data)

total = sum(vt_counter.values())
num_classes = len(vt_labels)

weights = []

for label in vt_labels:
    count = vt_counter[label]
    weight = total / (num_classes * count)
    weights.append(weight)

weights = torch.tensor(weights, dtype=torch.float).to(device)
weights = torch.clamp(weights, max=5.0)

print("VT Counter:", vt_counter)
print("VT weights:", weights)

timeline_criterions = {
    "verification_timeline": FocalLoss(
        weight=weights,
        gamma=2.0
    )
}

timeline_task_weights = {
    "verification_timeline": 1.0
}
# =====================================================
# 5. Stratified 5-fold by VT
# =====================================================

vt_y = [
    expert_label2id["verification_timeline"][x["verification_timeline"]]
    for x in all_data
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

timeline_folds = []

for fold_id, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(all_data)), vt_y)
):
    train_fold = [all_data[i] for i in train_idx]
    val_fold = [all_data[i] for i in val_idx]

    timeline_folds.append((train_fold, val_fold))

    print(f"Fold {fold_id+1}")
    print("Train:", Counter(x["verification_timeline"] for x in train_fold))
    print("Val  :", Counter(x["verification_timeline"] for x in val_fold))
# =====================================================
# 6. 建立 loader
# =====================================================

def build_timeline_testsafe_loaders(train_fold, val_fold):
    train_ds = ESGTimelineDataset_TestSafe(
        train_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    val_ds = ESGTimelineDataset_TestSafe(
        val_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    return train_loader, val_loader

NameError: name 'vt_labels' is not defined

In [ ]:
import re
import os
import json
import torch
import pandas as pd
import numpy as np

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold

# =====================================================
# 2. TestSafe Timeline Feature
# =====================================================

def extract_raw_evidence(text):
    patterns = (
        r'(\d+(\.\d+)?%|'
        r'ISO\s*\d+|'
        r'\d{4}年|'
        r'\d+\s*(?:公噸|公斤|度|kWh|MW|億元|萬元|天|週|年))'
    )

    matches = re.findall(patterns, str(text))

    found = [
        m[0] if isinstance(m, tuple) else m
        for m in matches
    ]

    return "/".join(found[:5]) if found else "None"


def extract_target_year(text):
    years = re.findall(r"20\d{2}", str(text))

    if len(years) == 0:
        return -1

    years = [int(y) for y in years]

    future_years = [y for y in years if y >= 2024]

    if len(future_years) == 0:
        return max(years)

    return min(future_years)


def build_timeline_prefix_testsafe(item):
    text = str(item.get("data", ""))

    target_year = extract_target_year(text)

    if target_year == -1:
        year_diff = -1
        year_bucket = "UNK"
    else:
        year_diff = target_year - 2024

        if year_diff <= 0:
            year_bucket = "ALREADY"
        elif year_diff <= 2:
            year_bucket = "SHORT"
        elif year_diff <= 5:
            year_bucket = "MID"
        else:
            year_bucket = "LONG"

    completed_words = [
        "已完成", "已達成", "已取得", "已建置",
        "已導入", "通過", "完成", "截至"
    ]

    annual_words = [
        "每年", "年度", "逐年", "定期", "每季",
        "每月", "持續追蹤", "持續檢討"
    ]

    continue_words = [
        "持續", "長年", "逐步", "規劃",
        "推動", "落實", "深化", "精進"
    ]

    short_words = [
        "短期", "近期", "兩年內", "2025", "2026"
    ]

    mid_words = [
        "中期", "五年內", "2027", "2028", "2029", "2030"
    ]

    long_words = [
        "長期", "長遠", "永續經營", "淨零",
        "2050", "2040"
    ]

    has_completed = int(any(w in text for w in completed_words))
    has_annual = int(any(w in text for w in annual_words))
    has_continue = int(any(w in text for w in continue_words))
    has_short = int(any(w in text for w in short_words))
    has_mid = int(any(w in text for w in mid_words))
    has_long = int(any(w in text for w in long_words))

    has_number = int(bool(re.search(r"\d", text)))
    has_year = int(bool(re.search(r"20\d{2}", text)))
    has_percent = int(bool(re.search(r"%|％", text)))

    raw_time = extract_raw_evidence(text)

    company = item.get("company", "UNK")
    ticker = item.get("ticker", "UNK")
    page_number = item.get("page_number", "UNK")

    return (
        f"[COMPANY={company}] "
        f"[TICKER={ticker}] "
        f"[PAGE={page_number}] "
        f"[NUM={has_number}] "
        f"[YEAR={has_year}] "
        f"[PERCENT={has_percent}] "
        f"[TARGET_YEAR={target_year}] "
        f"[YEAR_DIFF={year_diff}] "
        f"[YEAR_BUCKET={year_bucket}] "
        f"[HAS_COMPLETED={has_completed}] "
        f"[HAS_ANNUAL={has_annual}] "
        f"[HAS_CONTINUE={has_continue}] "
        f"[HAS_SHORT={has_short}] "
        f"[HAS_MID={has_mid}] "
        f"[HAS_LONG={has_long}] "
        f"[TIME={raw_time}] "
    )
# =====================================================
# 3. TestSafe Timeline Dataset
# =====================================================

class ESGTimelineDataset_TestSafe(Dataset):

    def __init__(self, data, tokenizer, label2id, fields):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.fields = fields

    def __len__(self):
        return len(self.data)

    def build_text(self, item):
        prefix = build_timeline_prefix_testsafe(item)
        text = str(item.get("data", ""))

        return prefix + "[TEXT] " + text

    def __getitem__(self, idx):
        item = self.data[idx]
        text = self.build_text(item)

        encoding = tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        labels = {}

        for field in self.fields:
            mapping = self.label2id[field]

            if field in item:
                label = item[field]

                if field == "verification_timeline" and label == "longer_than_5_years":
                    label = "more_than_5_years"

                labels[field] = mapping[label]
            else:
                labels[field] = -1

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": labels,
        }
# =====================================================
# 4. 重建 FocalLoss weight
# =====================================================

vt_counter = Counter(x["verification_timeline"] for x in all_data)

total = sum(vt_counter.values())

classes = list(
    expert_label2id["verification_timeline"].keys()
)

num_classes = len(classes)

weights = []

for label in classes:

    count = vt_counter.get(label, 1)

    weight = total / (num_classes * count)

    weights.append(weight)

weights = torch.tensor(
    weights,
    dtype=torch.float
).to(device)

weights = torch.clamp(
    weights,
    max=5.0
)

timeline_criterions_testsafe = {
    "verification_timeline": FocalLoss(
        weight=weights,
        gamma=1.5
    )
}

timeline_task_weights = {
    "verification_timeline": 1.0
}
# =====================================================
# 5. Stratified 5-fold by VT
# =====================================================

vt_y = [
    expert_label2id["verification_timeline"][x["verification_timeline"]]
    for x in all_data
]

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

timeline_folds = []

for fold_id, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(all_data)), vt_y)
):
    train_fold = [all_data[i] for i in train_idx]
    val_fold = [all_data[i] for i in val_idx]

    timeline_folds.append((train_fold, val_fold))

    print(f"Fold {fold_id+1}")
    print("Train:", Counter(x["verification_timeline"] for x in train_fold))
    print("Val  :", Counter(x["verification_timeline"] for x in val_fold))
# =====================================================
# 6. 建立 loader
# =====================================================

def build_timeline_testsafe_loaders(train_fold, val_fold):
    train_ds = ESGTimelineDataset_TestSafe(
        train_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    val_ds = ESGTimelineDataset_TestSafe(
        val_fold,
        tokenizer,
        expert_label2id,
        timeline_fields
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn_factory(timeline_fields)
    )

    return train_loader, val_loader

Fold 1
Train: Counter({'already': 574, 'between_2_and_5_years': 398, 'more_than_5_years': 301, 'N/A': 299, 'within_2_years': 28})
Val  : Counter({'already': 144, 'between_2_and_5_years': 100, 'more_than_5_years': 76, 'N/A': 74, 'within_2_years': 6})
Fold 2
Train: Counter({'already': 574, 'between_2_and_5_years': 399, 'more_than_5_years': 302, 'N/A': 298, 'within_2_years': 27})
Val  : Counter({'already': 144, 'between_2_and_5_years': 99, 'more_than_5_years': 75, 'N/A': 75, 'within_2_years': 7})
Fold 3
Train: Counter({'already': 574, 'between_2_and_5_years': 399, 'more_than_5_years': 302, 'N/A': 298, 'within_2_years': 27})
Val  : Counter({'already': 144, 'between_2_and_5_years': 99, 'more_than_5_years': 75, 'N/A': 75, 'within_2_years': 7})
Fold 4
Train: Counter({'already': 575, 'between_2_and_5_years': 398, 'more_than_5_years': 302, 'N/A': 298, 'within_2_years': 27})
Val  : Counter({'already': 143, 'between_2_and_5_years': 100, 'more_than_5_years': 75, 'N/A': 75, 'within_2_years': 7})
Fo

In [ ]:
# =====================================================
# 6. 訓練 Timeline TestSafe 5-fold
# =====================================================
import gc
SAVE_DIR_VT_TESTSAFE = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v4"
os.makedirs(SAVE_DIR_VT_TESTSAFE, exist_ok=True)
EPOCHS = 10
timeline_testsafe_scores = []

for fold_id in range(5):

    print(f"\n========== Timeline TestSafe Fold {fold_id+1}/5 ==========")

    train_fold, val_fold = timeline_folds[fold_id]

    train_loader, val_loader = build_timeline_testsafe_loaders(
        train_fold,
        val_fold
    )

    if fold_id == 0:
        ds = train_loader.dataset
        print("\n[TestSafe sample]")
        print(ds.build_text(ds.data[0])[:500])

    model_timeline = TimelineGuardian(
        expert_label_counts
    ).to(device)

    optimizer_timeline = AdamW(
        model_timeline.parameters(),
        lr=2e-5
    )

    scheduler_timeline = get_linear_schedule_with_warmup(
        optimizer_timeline,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * EPOCHS
    )

    save_path = (
        f"{SAVE_DIR_VT_TESTSAFE}/"
        f"best_timeline_testsafe_fold{fold_id+1}.pt"
    )

    best_vt = run_expert_training(
        expert_name=f"Timeline_TestSafe_Fold{fold_id+1}",
        model=model_timeline,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer_timeline,
        scheduler=scheduler_timeline,
        criterions=timeline_criterions_testsafe,
        task_weights=timeline_task_weights,
        fields=timeline_fields,
        save_path=save_path,
        alpha=1,
        device=device
    )

    timeline_testsafe_scores.append({
        "fold": fold_id + 1,
        "timeline_testsafe": best_vt
    })

    print(f"✅ Fold {fold_id+1} done | VT={best_vt:.5f}")

    del model_timeline
    gc.collect()
    torch.cuda.empty_cache()

timeline_testsafe_score_df = pd.DataFrame(
    timeline_testsafe_scores
)

print(timeline_testsafe_score_df)

print(
    "Timeline TestSafe Mean:",
    timeline_testsafe_score_df["timeline_testsafe"].mean()
)


========== Timeline TestSafe Fold 1/5 ==========

[TestSafe sample]
[COMPANY=tcc] [TICKER=1101] [PAGE=32] [NUM=0] [YEAR=0] [PERCENT=0] [TARGET_YEAR=-1] [YEAR_DIFF=-1] [YEAR_BUCKET=UNK] [HAS_COMPLETED=0] [HAS_ANNUAL=0] [HAS_CONTINUE=1] [HAS_SHORT=0] [HAS_MID=0] [HAS_LONG=0] [TIME=None] [TEXT] 在與供應商攜手邁向永續發展的過程中，台泥致力於建立一套以合作為基礎的價值體體系。核心原則是推動供應鏈的低碳轉型，同時維護人權、保護環境和促進生物多樣性，以建立永續的合作夥伴關係。


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Timeline_TestSafe_Fold1！ (R-Drop alpha=1)

[Timeline_TestSafe_Fold1] Epoch 1/12
----------------------------------------
  平均 Loss: 0.8004 | 加權得分: 0.06386
  -> verification_timeline : Macro F1 = 0.4258
  ✅ 最佳模型已儲存！得分: 0.06386

[Timeline_TestSafe_Fold1] Epoch 2/12
----------------------------------------
  平均 Loss: 0.6036 | 加權得分: 0.07102
  -> verification_timeline : Macro F1 = 0.4734
  ✅ 最佳模型已儲存！得分: 0.07102

[Timeline_TestSafe_Fold1] Epoch 3/12
----------------------------------------
  平均 Loss: 0.4288 | 加權得分: 0.08720
  -> verification_timeline : Macro F1 = 0.5814
  ✅ 最佳模型已儲存！得分: 0.08720

[Timeline_TestSafe_Fold1] Epoch 4/12
----------------------------------------
  平均 Loss: 0.2902 | 加權得分: 0.08614
  -> verification_timeline : Macro F1 = 0.5743

[Timeline_TestSafe_Fold1] Epoch 5/12
----------------------------------------
  平均 Loss: 0.1853 | 加權得分: 0.08846
  -> verification_timeline : Macro F1 = 0.5898
  ✅ 最佳模型已儲存！得分: 0.08846

[Timeline_TestSafe_Fold1] Epoch 6/12
----------------

NameError: name 'gc' is not defined

In [ ]:
# 1. 確認 VT loss
print("timeline_criterions =", timeline_criterions)
print("VT loss type =", type(timeline_criterions["verification_timeline"]))
print("VT loss dict =", timeline_criterions["verification_timeline"].__dict__)

# 2. 確認 VT label mapping
print("label2id VT =", expert_label2id["verification_timeline"])
print("id2label VT =", expert_id2label["verification_timeline"])

# 3. 確認 train 資料 label 分布
from collections import Counter
print("train VT labels:")
print(Counter([x["verification_timeline"] for x in train_fold]))

print("val VT labels:")
print(Counter([x["verification_timeline"] for x in val_fold]))

# 4. 確認 Timeline Dataset 真的用舊版 prefix
print(timeline_val_ds.build_text(timeline_val_ds.data[0])[:300])

# 5. 確認現在用的是哪個 Timeline model class
print(model_timeline)

timeline_criterions = {'verification_timeline': FocalLoss()}
VT loss type = <class '__main__.FocalLoss'>
VT loss dict = {'training': True, '_parameters': {}, '_buffers': {}, '_non_persistent_buffers_set': set(), '_backward_pre_hooks': OrderedDict(), '_backward_hooks': OrderedDict(), '_is_full_backward_hook': None, '_forward_hooks': OrderedDict(), '_forward_hooks_with_kwargs': OrderedDict(), '_forward_hooks_always_called': OrderedDict(), '_forward_pre_hooks': OrderedDict(), '_forward_pre_hooks_with_kwargs': OrderedDict(), '_state_dict_hooks': OrderedDict(), '_state_dict_pre_hooks': OrderedDict(), '_load_state_dict_pre_hooks': OrderedDict(), '_load_state_dict_post_hooks': OrderedDict(), '_modules': {}, 'weight': tensor([0.5571, 8.0000, 0.8032, 1.0610, 1.0724], device='cuda:0'), 'gamma': 2.0, 'reduction': 'mean'}
label2id VT = {'already': 0, 'within_2_years': 1, 'between_2_and_5_years': 2, 'more_than_5_years': 3, 'N/A': 4}
id2label VT = {0: 'already', 1: 'within_2_years', 2: 'between_2_an

In [ ]:
import torch

FOLD_DIR = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v2"
FOLD_DIR2 = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v2_timelimeOLD"
FOLD_DIR3 = "/content/drive/MyDrive/AI_CUP_ESG/5fold_timeline_testsafe"
anchor_models = []
quality_models = []
timeline_models = []

for fold in range(1, 6):

   # Anchor
   m_anchor = AnchorModel(expert_label_counts).to(device)
   m_anchor.load_state_dict(
       torch.load(f"{FOLD_DIR}/best_anchor_fold{fold}.pt", map_location=device)
   )
   m_anchor.eval()
   anchor_models.append(m_anchor)

   # Quality
   m_quality = QualitySharedExpert(expert_label_counts).to(device)
   m_quality.load_state_dict(
       torch.load(f"{FOLD_DIR2}/best_quality_fgm_fold{fold}.pt", map_location=device)
   )
   m_quality.eval()
   quality_models.append(m_quality)

   # Timeline
   m_timeline = TimelineGuardian(expert_label_counts).to(device)
   m_timeline.load_state_dict(
       torch.load(f"{FOLD_DIR2}/best_timeline_fold_new{fold}.pt", map_location=device)
   )
   m_timeline.eval()
   timeline_models.append(m_timeline)

print("✅ 5-fold models loaded")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ 5-fold models loaded


#2. 5-fold Ensemble Inference

In [ ]:
anchor_models = load_anchor_models()
quality_models_fgm = load_quality_fgm_models()
quality_models_nofgm = load_quality_nofgm_models()
timeline_models_v2 = load_timeline_v2_models()
timeline_models_testsafe = load_timeline_testsafe_models()

NameError: name 'load_anchor_models' is not defined

In [ ]:
# =====================================================
# Validation Dataset
# =====================================================

anchor_val_1000_ds = ESGDataset(
    val_data_1000,
    tokenizer,
    expert_label2id,
    anchor_fields,
    mode="anchor"
)

quality_val_1000_ds = ESGDataset(
    val_data_1000,
    tokenizer,
    expert_label2id,
    quality_fields,
    mode="quality"
)

# =====================================================
# Timeline FeatureV2
# =====================================================

timeline_val_1000_ds_v2 = ESGDataset_featurev2(
    val_data_1000,
    tokenizer,
    expert_label2id,
    timeline_fields
)

# =====================================================
# Timeline TestSafe
# =====================================================

timeline_val_1000_ds_testsafe = ESGTimelineDataset_TestSafe(
    val_data_1000,
    tokenizer,
    expert_label2id,
    timeline_fields
)

# =====================================================
# Loaders
# =====================================================

anchor_val_1000_loader = DataLoader(
    anchor_val_1000_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(anchor_fields)
)

quality_val_1000_loader = DataLoader(
    quality_val_1000_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(quality_fields)
)

timeline_val_1000_loader_v2 = DataLoader(
    timeline_val_1000_ds_v2,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(timeline_fields)
)

timeline_val_1000_loader_testsafe = DataLoader(
    timeline_val_1000_ds_testsafe,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(timeline_fields)
)

# =====================================================
# FeatureV2 Loader
# =====================================================

expert_loaders_1000_v2 = {
    "anchor": anchor_val_1000_loader,
    "quality": quality_val_1000_loader,
    "timeline": timeline_val_1000_loader_v2
}

# =====================================================
# TestSafe Loader
# =====================================================

expert_loaders_1000_testsafe = {
    "anchor": anchor_val_1000_loader,
    "quality": quality_val_1000_loader,
    "timeline": timeline_val_1000_loader_testsafe
}

print("✅ expert_loaders_1000_v2 建立完成")
print("✅ expert_loaders_1000_testsafe 建立完成")

print("\n[V2 Sample]")
print(
    timeline_val_1000_ds_v2.build_text(
        timeline_val_1000_ds_v2.data[0]
    )[:500]
)

print("\n[TestSafe Sample]")
print(
    timeline_val_1000_ds_testsafe.build_text(
        timeline_val_1000_ds_testsafe.data[0]
    )[:500]
)

✅ expert_loaders_1000_v2 建立完成
✅ expert_loaders_1000_testsafe 建立完成

[V2 Sample]
[ESG=S] [NUM=1] [YEAR=0] [PERCENT=0] [VAGUE=3] [VAGUE_HIGH=1] [CLEAR=0] [VT_ALREADY=0] [VT_SHORT=0] [VT_MID=0] [VT_LONG=0] [RAW_EVD=None] 為有效預防控制職業危害，公司制訂有「職業病控制管理規定」。公司對所涉及的職業病危害項目由營運服務部向政府部門進行申報，由有資質的技術服務機構提供評價工作，並獲得相關部門的驗收批復。根據危險辨識與控制內容對從事接觸職業病危害因素工作的員工進行職業培訓及崗前、崗中、崗後職業健康檢查。  按照GRI準則，以失能傷害頻率(Disabling Injury Frequency Rate，簡稱FR)、失能傷害嚴重率(Disabling Injury Severity Rat，簡稱SR)作為主要數據指標揭露重工傷情況，本報告期內，員工失能傷害6件，輕工傷40件，主要是運行機台作業過程中所致壓傷、夾傷和摔傷、碰傷等。對於職業傷害，公司均落實改善措施，包括：增設設備安全防護，嚴格落實設備點檢與保養，加強安全教育培訓，管理人員高頻巡檢

[TestSafe Sample]
[COMPANY=ltc] [TICKER=2301] [PAGE=109] [NUM=1] [YEAR=0] [PERCENT=0] [TARGET_YEAR=-1] [YEAR_DIFF=-1] [YEAR_BUCKET=UNK] [HAS_COMPLETED=0] [HAS_ANNUAL=0] [HAS_CONTINUE=1] [HAS_SHORT=0] [HAS_MID=0] [HAS_LONG=0] [TIME=None] [TEXT] 為有效預防控制職業危害，公司制訂有「職業病控制管理規定」。公司對所涉及的職業病危害項目由營運服務部向政府部門進行申報，由有資質的技術服務機構提供評價工作，並獲得相關部門的驗收批復。根據危險辨識與控制內容對從事接觸職業病危害因素工作的員工進行職業培訓及崗前、崗中、崗後職業健康檢查。  按照GRI準則，以失能傷害頻率(Disabling Injury 

In [ ]:
import gc
import torch

# 刪掉大型模型清單
for name in [
    "anchor_models",
    "quality_models",
    "quality_models_fgm",
    "quality_models_nofgm",
    "timeline_models",
    "timeline_models_v2",
    "timeline_models_testsafe",
    "model_anchor",
    "model_quality",
    "model_timeline"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 5            |        cudaMalloc retries: 6         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4470 MiB |  21470 MiB | 399431 GiB | 399426 GiB |
|       from large pool |   4455 MiB |  21398 MiB | 399196 GiB | 399192 GiB |
|       from small pool |     15 MiB |     72 MiB |    234 GiB |    234 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   4470 MiB |  21470 MiB | 399431 GiB | 399426 GiB |
|       from large pool |   4455 MiB |  21398 MiB | 399196 GiB |

##載入權重檔

---



In [ ]:
# =====================================================
# 1. Quality Ensemble + VT Ensemble Inference
# =====================================================
import os
import torch
from collections import Counter

# =====================================================
# 1. 載入 Quality NoFGM / FGM 5-fold models
# =====================================================
#SAVE_DIR = "/content/drive/MyDrive/AI_CUP_ESG/5fold_v4"
ANCHOR_DIR = "weights/anchor"
QUALITY_DIR = "weights/quality"
TIMELINE_DIR = "weights/timeline"


anchor_models = []
quality_models_nofgm = []
quality_models_fgm = []

timeline_models_v2 = []
timeline_models_testsafe = []
for fold_id in range(1, 6):




    m_anchor = AnchorModel(
        expert_label_counts
    ).to(device)

    m_anchor.load_state_dict(
        torch.load(
            f"{ANCHOR_DIR}/best_anchor_fold{fold_id}.pt",
            map_location=device
        )
    )

    m_anchor.eval()
    anchor_models.append(m_anchor)


    # No FGM
    model_nofgm = QualitySharedExpert(
        expert_label_counts
    ).to(device)

    model_nofgm.load_state_dict(
        torch.load(
            f"{QUALITY_DIR}/best_quality_fold{fold_id}.pt",
            map_location=device
        )
    )

    model_nofgm.eval()
    quality_models_nofgm.append(model_nofgm)

    # ES FGM
    model_fgm = QualitySharedExpert(
        expert_label_counts
    ).to(device)

    model_fgm.load_state_dict(
        torch.load(
            f"{QUALITY_DIR}/best_quality_fgm_esonly_fold{fold_id}.pt",
            map_location=device
        )
    )

    model_fgm.eval()
    quality_models_fgm.append(model_fgm)



    m_timeline = TimelineGuardian(
        expert_label_counts
    ).to(device)

    m_timeline.load_state_dict(
        torch.load(
            f"{TIMELINE_DIR}/best_timeline_fold_new{fold_id}.pt",
            map_location=device
        )
    )

    m_timeline.eval()
    timeline_models_v2.append(m_timeline)

    model_testafe = TimelineGuardian(
        expert_label_counts
    ).to(device)

    model_testafe.load_state_dict(
        torch.load(
            f"{TIMELINE_DIR}/best_timeline_testsafe_fold{fold_id}.pt",
            map_location=device
        )
    )

    model_testafe.eval()
    timeline_models_testsafe.append(model_testafe)

print("PS models:", len(anchor_models))
print("NoFGM Quality models:", len(quality_models_nofgm))
print("FGM Quality models:", len(quality_models_fgm))
print("VT v2 models:", len(timeline_models_v2))
print("VT testsafe models:", len(timeline_models_testsafe))
def expert_inference_quality_vt_ensemble(
    anchor_models,
    quality_models_fgm,
    quality_models_nofgm,
    timeline_models_v2,
    timeline_models_testsafe,
    loaders_v2,
    loaders_testsafe,
    device,
    id2label,
    no_threshold=0.7,
    es_fgm_weight=0.7,
    eq_fgm_weight=0.3,
    vt_v2_weight=0.8
):
    """
    Quality Ensemble:
      ES = es_fgm_weight * FGM + (1 - es_fgm_weight) * NoFGM
      EQ = eq_fgm_weight * FGM + (1 - eq_fgm_weight) * NoFGM

    VT Ensemble:
      VT = vt_v2_weight * FeatureV2 + (1 - vt_v2_weight) * TestSafe

    loaders_v2:
      anchor / quality / timeline 用 FeatureV2 timeline loader

    loaders_testsafe:
      只使用 timeline 的 TestSafe loader
    """

    vt_testsafe_weight = 1 - vt_v2_weight

    for model_list in [
        anchor_models,
        quality_models_fgm,
        quality_models_nofgm,
        timeline_models_v2,
        timeline_models_testsafe
    ]:
        for m in model_list:
            m.eval()

    all_preds = []

    anchor_loader = loaders_v2["anchor"]
    quality_loader = loaders_v2["quality"]

    timeline_loader_v2 = loaders_v2["timeline"]
    timeline_loader_testsafe = loaders_testsafe["timeline"]

    no_id = None
    for k, v in id2label["promise_status"].items():
        if v == "No":
            no_id = k
            break

    with torch.no_grad():

        for (
            anchor_batch,
            quality_batch,
            timeline_batch_v2,
            timeline_batch_testsafe
        ) in zip(
            anchor_loader,
            quality_loader,
            timeline_loader_v2,
            timeline_loader_testsafe
        ):

            # =========================
            # Inputs
            # =========================

            anchor_input_ids = anchor_batch["input_ids"].to(device)
            anchor_attention_mask = anchor_batch["attention_mask"].to(device)

            quality_input_ids = quality_batch["input_ids"].to(device)
            quality_attention_mask = quality_batch["attention_mask"].to(device)

            vt_v2_input_ids = timeline_batch_v2["input_ids"].to(device)
            vt_v2_attention_mask = timeline_batch_v2["attention_mask"].to(device)

            vt_ts_input_ids = timeline_batch_testsafe["input_ids"].to(device)
            vt_ts_attention_mask = timeline_batch_testsafe["attention_mask"].to(device)

            # =========================
            # PS Ensemble
            # =========================

            ps_logits_sum = None

            for m in anchor_models:
                out = m(anchor_input_ids, anchor_attention_mask)
                logits = out["promise_status"]

                ps_logits_sum = (
                    logits
                    if ps_logits_sum is None
                    else ps_logits_sum + logits
                )

            ps_logits_avg = ps_logits_sum / len(anchor_models)
            ps_probs = torch.softmax(ps_logits_avg, dim=-1)

            # =========================
            # Quality FGM Ensemble
            # =========================

            es_fgm_sum = None
            eq_fgm_sum = None

            for m in quality_models_fgm:
                out = m(quality_input_ids, quality_attention_mask)

                es_logits = out["evidence_status"]
                eq_logits = out["evidence_quality"]

                es_fgm_sum = (
                    es_logits
                    if es_fgm_sum is None
                    else es_fgm_sum + es_logits
                )

                eq_fgm_sum = (
                    eq_logits
                    if eq_fgm_sum is None
                    else eq_fgm_sum + eq_logits
                )

            es_fgm_avg = es_fgm_sum / len(quality_models_fgm)
            eq_fgm_avg = eq_fgm_sum / len(quality_models_fgm)

            # =========================
            # Quality NoFGM Ensemble
            # =========================

            es_nofgm_sum = None
            eq_nofgm_sum = None

            for m in quality_models_nofgm:
                out = m(quality_input_ids, quality_attention_mask)

                es_logits = out["evidence_status"]
                eq_logits = out["evidence_quality"]

                es_nofgm_sum = (
                    es_logits
                    if es_nofgm_sum is None
                    else es_nofgm_sum + es_logits
                )

                eq_nofgm_sum = (
                    eq_logits
                    if eq_nofgm_sum is None
                    else eq_nofgm_sum + eq_logits
                )

            es_nofgm_avg = es_nofgm_sum / len(quality_models_nofgm)
            eq_nofgm_avg = eq_nofgm_sum / len(quality_models_nofgm)

            # =========================
            # ES / EQ Logits Ensemble
            # =========================

            es_logits_avg = (
                es_fgm_weight * es_fgm_avg
                + (1 - es_fgm_weight) * es_nofgm_avg
            )

            eq_logits_avg = (
                eq_fgm_weight * eq_fgm_avg
                + (1 - eq_fgm_weight) * eq_nofgm_avg
            )

            # =========================
            # VT FeatureV2 Ensemble
            # =========================

            vt_v2_sum = None

            for m in timeline_models_v2:
                out = m(vt_v2_input_ids, vt_v2_attention_mask)
                logits = out["verification_timeline"]

                vt_v2_sum = (
                    logits
                    if vt_v2_sum is None
                    else vt_v2_sum + logits
                )

            vt_v2_avg = vt_v2_sum / len(timeline_models_v2)

            # =========================
            # VT TestSafe Ensemble
            # =========================

            vt_ts_sum = None

            for m in timeline_models_testsafe:
                out = m(vt_ts_input_ids, vt_ts_attention_mask)
                logits = out["verification_timeline"]

                vt_ts_sum = (
                    logits
                    if vt_ts_sum is None
                    else vt_ts_sum + logits
                )

            vt_ts_avg = vt_ts_sum / len(timeline_models_testsafe)

            # =========================
            # VT Logits Ensemble
            # =========================

            vt_logits_avg = (
                vt_v2_weight * vt_v2_avg
                + vt_testsafe_weight * vt_ts_avg
            )

            # =========================
            # Decode
            # =========================

            batch_size = anchor_input_ids.size(0)

            for i in range(batch_size):

                ps_pred_id = ps_logits_avg[i].argmax().item()
                ps_pred = id2label["promise_status"][ps_pred_id]
                ps_no_prob = ps_probs[i][no_id].item()

                es_pred = id2label["evidence_status"][
                    es_logits_avg[i].argmax().item()
                ]

                eq_pred = id2label["evidence_quality"][
                    eq_logits_avg[i].argmax().item()
                ]

                vt_pred = id2label["verification_timeline"][
                    vt_logits_avg[i].argmax().item()
                ]

                if vt_pred == "longer_than_5_years":
                    vt_pred = "more_than_5_years"

                # Soft Gate
                if (
                    no_threshold is not None
                    and ps_pred == "No"
                    and ps_no_prob >= no_threshold
                ):
                    final_item = {
                        "promise_status": "No",
                        "verification_timeline": "N/A",
                        "evidence_status": "N/A",
                        "evidence_quality": "N/A"
                    }

                else:
                    final_item = {
                        "promise_status": ps_pred,
                        "verification_timeline": vt_pred,
                        "evidence_status": es_pred,
                        "evidence_quality": eq_pred
                    }

                all_preds.append(final_item)

    return all_preds



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PS models: 5
NoFGM Quality models: 5
FGM Quality models: 5
VT v2 models: 5
VT testsafe models: 5


In [ ]:
# =====================================================
# 2. 掃 VT ensemble 權重 + Gate threshold
# =====================================================


vt_weights = [
0.35,
0.40,
0.45


]

thresholds = [

    0.80,
    0.70




]

best_score = -1
best_setting = None
best_results = None
best_preds = None

for vt_w in vt_weights:

    print("\n" + "=" * 70)
    print(f"開始測 VT Ensemble | FeatureV2={vt_w:.2f}, TestSafe={1-vt_w:.2f}")
    print("=" * 70)

    for th in thresholds:

        preds = expert_inference_quality_vt_ensemble(
            anchor_models=anchor_models,
            quality_models_fgm=quality_models_fgm,
            quality_models_nofgm=quality_models_nofgm,
            timeline_models_v2=timeline_models_v2,
            timeline_models_testsafe=timeline_models_testsafe,
            loaders_v2=expert_loaders_1000_v2,
            loaders_testsafe=expert_loaders_1000_testsafe,
            device=device,
            id2label=expert_id2label,
            no_threshold=th,
            es_fgm_weight=0.8,
            eq_fgm_weight=0.2,
            vt_v2_weight=vt_w
        )

        results = evaluate_hybrid(
            val_gt_1000,
            preds
        )

        score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in FIELD_WEIGHTS
        )

        is_best = score > best_score

        if is_best:
            best_score = score
            best_setting = {
                "vt_v2_weight": vt_w,
                "vt_testsafe_weight": 1 - vt_w,
                "threshold": th
            }
            best_results = results
            best_preds = preds

        mark = "🔥 BEST" if is_best else ""

        print(
            f"VT_V2={vt_w:.2f} "
            f"TS={1-vt_w:.2f} "
            f"TH={th:.2f} | "
            f"PS={results['promise_status']['macro_f1']:.4f} | "
            f"ES={results['evidence_status']['macro_f1']:.4f} | "
            f"EQ={results['evidence_quality']['macro_f1']:.4f} | "
            f"VT={results['verification_timeline']['macro_f1']:.4f} | "
            f"SCORE={score:.5f} "
            f"{mark}"
        )

print("\n" + "=" * 70)
print("FINAL BEST")
print("=" * 70)

print(best_setting)
print(f"Best score: {best_score:.5f}")

for f in FIELD_WEIGHTS:
    print(f"{f:24}: {best_results[f]['macro_f1']:.4f}")

In [ ]:
quality_settings = [


    (0.6, 0.0),




]

best_score = -1

for es_w, eq_w in quality_settings:

    preds = expert_inference_quality_vt_ensemble(
        anchor_models=anchor_models,
        quality_models_fgm=quality_models_fgm,
        quality_models_nofgm=quality_models_nofgm,
        timeline_models_v2=timeline_models_v2,
        timeline_models_testsafe=timeline_models_testsafe,
        loaders_v2=expert_loaders_1000_v2,
        loaders_testsafe=expert_loaders_1000_testsafe,
        device=device,
        id2label=expert_id2label,
        no_threshold=0.70,
        es_fgm_weight=es_w,
        eq_fgm_weight=eq_w,
        vt_v2_weight=0.45
    )

    results = evaluate_hybrid(val_gt_1000, preds)

    score = sum(
        results[f]["macro_f1"] * FIELD_WEIGHTS[f]
        for f in FIELD_WEIGHTS
    )

    print(
        f"ES_FGM={es_w:.2f} EQ_FGM={eq_w:.2f} | "
        f"ES={results['evidence_status']['macro_f1']:.4f} | "
        f"EQ={results['evidence_quality']['macro_f1']:.4f} | "
        f"VT={results['verification_timeline']['macro_f1']:.4f} | "
        f"SCORE={score:.5f}"
    )

    if score > best_score:
        best_score = score
        best_quality_setting = (es_w, eq_w)
        best_quality_results = results

ES_FGM=1.00 EQ_FGM=0.00 | ES=0.6999 | EQ=0.4464 | VT=0.6170 | SCORE=0.62074


In [ ]:
quality_settings = [
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
]

thresholds = [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

best_score = -1
best_setting = None
best_results = None

for es_w, eq_w in quality_settings:
    for th in thresholds:

        preds = expert_inference_5fold_quality_ensemble_gate(
            anchor_models=anchor_models,
            quality_models_fgm=quality_models_fgm,
            quality_models_nofgm=quality_models_nofgm,
            timeline_models=timeline_models,
            loaders=expert_loaders_1000,
            device=device,
            id2label=expert_id2label,
            no_threshold=th,
            es_fgm_weight=es_w,
            eq_fgm_weight=eq_w
        )

        results = evaluate_hybrid(
            val_gt_1000,
            preds
        )

        score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in FIELD_WEIGHTS
        )

        print(
            f"ES_FGM={es_w:.1f} EQ_FGM={eq_w:.1f} TH={th:.2f} | "
            f"PS={results['promise_status']['macro_f1']:.4f} | "
            f"ES={results['evidence_status']['macro_f1']:.4f} | "
            f"EQ={results['evidence_quality']['macro_f1']:.4f} | "
            f"VT={results['verification_timeline']['macro_f1']:.4f} | "
            f"SCORE={score:.5f}"
        )

        if score > best_score:
            best_score = score
            best_setting = (es_w, eq_w, th)
            best_results = results

print("\nBest setting:", best_setting)
print("Best score:", best_score)

for f in FIELD_WEIGHTS:
    print(f"{f:24}: {best_results[f]['macro_f1']:.4f}")

In [ ]:
thresholds = [
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90, 0.95
]

best_score = -1
best_threshold = None
best_results = None
best_preds = None

for th in thresholds:

    preds = expert_inference_5fold(
        anchor_models=anchor_models,
        quality_models=quality_models,
        timeline_models=timeline_models,
        loaders=expert_loaders_1000,
        device=device,
        id2label=expert_id2label,
        no_threshold=th
    )

    results = evaluate_hybrid(
        val_gt_1000,
        preds
    )

    score = sum(
        results[f]["macro_f1"] * FIELD_WEIGHTS[f]
        for f in FIELD_WEIGHTS
    )

    print(
        f"TH={th:.2f} | "
        f"PS={results['promise_status']['macro_f1']:.4f} | "
        f"ES={results['evidence_status']['macro_f1']:.4f} | "
        f"EQ={results['evidence_quality']['macro_f1']:.4f} | "
        f"VT={results['verification_timeline']['macro_f1']:.4f} | "
        f"SCORE={score:.5f}"
    )

    if score > best_score:
        best_score = score
        best_threshold = th
        best_results = results
        best_preds = preds

print("\n==============================")
print(f"Best threshold = {best_threshold}")
print(f"Best score     = {best_score:.5f}")
print("==============================")

for f in FIELD_WEIGHTS:
    print(f"{f:24}: {best_results[f]['macro_f1']:.4f}")

TH=0.50 | PS=0.8100 | ES=0.6977 | EQ=0.4434 | VT=0.4644 | SCORE=0.59614
TH=0.55 | PS=0.8100 | ES=0.6945 | EQ=0.4406 | VT=0.4624 | SCORE=0.59393
TH=0.60 | PS=0.8100 | ES=0.6967 | EQ=0.4409 | VT=0.4638 | SCORE=0.59490
TH=0.65 | PS=0.8100 | ES=0.6956 | EQ=0.4429 | VT=0.4635 | SCORE=0.59524
TH=0.70 | PS=0.8100 | ES=0.6990 | EQ=0.4425 | VT=0.4643 | SCORE=0.59620
TH=0.75 | PS=0.8100 | ES=0.6980 | EQ=0.4401 | VT=0.4641 | SCORE=0.59506
TH=0.80 | PS=0.8100 | ES=0.6980 | EQ=0.4401 | VT=0.4641 | SCORE=0.59506
TH=0.85 | PS=0.8100 | ES=0.7004 | EQ=0.4404 | VT=0.4643 | SCORE=0.59591
TH=0.90 | PS=0.8100 | ES=0.7014 | EQ=0.4397 | VT=0.4621 | SCORE=0.59560
TH=0.95 | PS=0.8100 | ES=0.6994 | EQ=0.4381 | VT=0.4601 | SCORE=0.59417

Best threshold = 0.7
Best score     = 0.59620
promise_status          : 0.8100
evidence_status         : 0.6990
evidence_quality        : 0.4425
verification_timeline   : 0.4643


#3. 用 vpesg4k_val_1000 驗證

In [ ]:
import json
import pandas as pd
from torch.utils.data import DataLoader

# =====================================================
# 1. 讀取 vpesg4k_val_1000
# =====================================================

val_1000_path = "/content/drive/MyDrive/AI_CUP_ESG/vpesg4k_val_1000.csv"

val_df_1000 = pd.read_csv(val_1000_path)
val_data_1000 = val_df_1000.to_dict("records")

print("val_1000 筆數:", len(val_data_1000))
print(val_data_1000[0])

In [ ]:
# =====================================================
# 3. 建立 val_gt_1000
# =====================================================

val_gt_1000 = []

for item in val_data_1000:
    val_gt_1000.append({
        "promise_status": item["promise_status"],
        "verification_timeline": item["verification_timeline"],
        "evidence_status": item["evidence_status"],
        "evidence_quality": item["evidence_quality"]
    })

print(val_gt_1000[0])

{'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}


In [ ]:
# =====================================================
# 4. 建立 validation DataLoader
# Anchor / Quality 用原本 Dataset
# Timeline 如果要測 V3，就用 ESGTimelineDatasetV3
# =====================================================

anchor_val_1000_ds = ESGDataset(
    val_data_1000,
    tokenizer,
    expert_label2id,
    anchor_fields,
    mode="anchor"
)

quality_val_1000_ds = ESGDataset(
    val_data_1000,
    tokenizer,
    expert_label2id,
    quality_fields,
    mode="quality"
)

# Timeline V3
timeline_val_1000_ds = ESGDataset_featurev2(
    val_data_1000,
    tokenizer,
    expert_label2id,
    timeline_fields
)

anchor_val_1000_loader = DataLoader(
    anchor_val_1000_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(anchor_fields)
)

quality_val_1000_loader = DataLoader(
    quality_val_1000_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(quality_fields)
)

timeline_val_1000_loader = DataLoader(
    timeline_val_1000_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(timeline_fields)
)

expert_loaders_1000 = {
    "anchor": anchor_val_1000_loader,
    "quality": quality_val_1000_loader,
    "timeline": timeline_val_1000_loader
}

print("✅ expert_loaders_1000 重建完成")

print("\nTimeline sample:")
print(timeline_val_1000_ds.build_text(timeline_val_1000_ds.data[0])[:500])

✅ expert_loaders_1000 重建完成

Timeline sample:
[ESG=S] [NUM=1] [YEAR=0] [PERCENT=0] [VAGUE=3] [VAGUE_HIGH=1] [CLEAR=0] [VT_ALREADY=0] [VT_SHORT=0] [VT_MID=0] [VT_LONG=0] [RAW_EVD=None] 為有效預防控制職業危害，公司制訂有「職業病控制管理規定」。公司對所涉及的職業病危害項目由營運服務部向政府部門進行申報，由有資質的技術服務機構提供評價工作，並獲得相關部門的驗收批復。根據危險辨識與控制內容對從事接觸職業病危害因素工作的員工進行職業培訓及崗前、崗中、崗後職業健康檢查。  按照GRI準則，以失能傷害頻率(Disabling Injury Frequency Rate，簡稱FR)、失能傷害嚴重率(Disabling Injury Severity Rat，簡稱SR)作為主要數據指標揭露重工傷情況，本報告期內，員工失能傷害6件，輕工傷40件，主要是運行機台作業過程中所致壓傷、夾傷和摔傷、碰傷等。對於職業傷害，公司均落實改善措施，包括：增設設備安全防護，嚴格落實設備點檢與保養，加強安全教育培訓，管理人員高頻巡檢


In [ ]:

preds_5fold_val_1000 = expert_inference_5fold(
   anchor_models=anchor_models,
   quality_models=quality_models,
   timeline_models=timeline_models,
   loaders=expert_loaders_1000,
   device=device,
   id2label=expert_id2label,
   no_threshold=0.7
)


In [ ]:
results_5fold_1000 = evaluate_hybrid(
   val_gt_1000,
   preds_5fold_val_1000
)

score_5fold_1000 = sum(
   results_5fold_1000[f]["macro_f1"] * FIELD_WEIGHTS[f]
   for f in FIELD_WEIGHTS
)

print("=" * 60)

for f in FIELD_WEIGHTS:
   print(f"{f:24}: {results_5fold_1000[f]['macro_f1']:.4f}")

print("=" * 60)
print(f"5-fold Ensemble Score: {score_5fold_1000:.5f}")


promise_status          : 0.8100
evidence_status         : 0.7014
evidence_quality        : 0.4397
verification_timeline   : 0.4822
5-fold Ensemble Score: 0.59861


In [ ]:
print("Pred 數量:", len(preds_5fold_val_1000))
print(preds_5fold_val_1000[:5])

Pred 數量: 1000
[{'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Not Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'N/A', 'evidence_quality': 'N/A'}]


##用test資料集


In [ ]:
import json
import pandas as pd
from torch.utils.data import DataLoader

# =====================================================
# 1. 讀取 test 2000
# =====================================================

test_path = "/content/drive/MyDrive/AI_CUP_ESG/vpesg4k_test_2000.json"

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Test 筆數:", len(test_data))
print(test_data[0])




Test 筆數: 2000
{'id': '12001', 'data': '工作、健康與家庭生活的平衡是多數人所追求的目標，瑞昱亦相當重視提升同仁的工作生活平衡與健康認知，因此長年關注與持續規劃執行涵蓋全體同仁 (包含約聘同仁) 之相關促進方案。例如：建置各類社團、設置瑞昱元氣活力館、悅讀小棧及悅讀館圖書室、舉辦各項團隊創新活力的精彩活動與運動賽事、開設多元主題的瑞昱知性生活系列講座，以及定期提供健康促進與疾病預防相關講座與資訊分享。在同仁的生活福利方面，公司亦逐年規劃多元方案照護同仁健康和提昇工作士氣、創造健康活力工作的職場文化，打造企業永續經營的健康體質與長遠競爭力。', 'company': 'rt', 'ticker': '2379', 'page_number': '136', 'pdf_url': 'https://www.realtek.com/images/csr/2024.pdf', 'company_source': 'https://www.realtek.com/Article/Index?menu_id=912&lang=zh-TW'}


In [ ]:
# =====================================================
# 2. 建立 test DataLoader
# Anchor / Quality 共用
# Timeline 分成 FeatureV2 與 TestSafe
# =====================================================

anchor_test_ds = ESGDataset(
    test_data,
    tokenizer,
    expert_label2id,
    anchor_fields,
    mode="anchor"
)

quality_test_ds = ESGDataset(
    test_data,
    tokenizer,
    expert_label2id,
    quality_fields,
    mode="quality"
)

# Timeline FeatureV2
timeline_test_ds_v2 = ESGDataset_featurev2(
    test_data,
    tokenizer,
    expert_label2id,
    timeline_fields,
    mode="timeline"
)

# Timeline TestSafe
timeline_test_ds_testsafe = ESGTimelineDataset_TestSafe(
    test_data,
    tokenizer,
    expert_label2id,
    timeline_fields
)

anchor_test_loader = DataLoader(
    anchor_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(anchor_fields)
)

quality_test_loader = DataLoader(
    quality_test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(quality_fields)
)

timeline_test_loader_v2 = DataLoader(
    timeline_test_ds_v2,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(timeline_fields)
)

timeline_test_loader_testsafe = DataLoader(
    timeline_test_ds_testsafe,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn_factory(timeline_fields)
)

# FeatureV2 loaders
expert_test_loaders_v2 = {
    "anchor": anchor_test_loader,
    "quality": quality_test_loader,
    "timeline": timeline_test_loader_v2
}

# TestSafe loaders
expert_test_loaders_testsafe = {
    "anchor": anchor_test_loader,
    "quality": quality_test_loader,
    "timeline": timeline_test_loader_testsafe
}

print("✅ expert_test_loaders_v2 建立完成")
print("✅ expert_test_loaders_testsafe 建立完成")

print("\n[Anchor sample]")
print(anchor_test_ds.build_text(anchor_test_ds.data[0])[:300])

print("\n[Timeline FeatureV2 sample]")
print(timeline_test_ds_v2.build_text(timeline_test_ds_v2.data[0])[:500])

print("\n[Timeline TestSafe sample]")
print(timeline_test_ds_testsafe.build_text(timeline_test_ds_testsafe.data[0])[:500])

✅ expert_test_loaders_v2 建立完成
✅ expert_test_loaders_testsafe 建立完成

[Anchor sample]
[ESG=UNK] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=3] [CLEAR=1] [TIMELINE=0/0/0] [RAW_EVD=None] 工作、健康與家庭生活的平衡是多數人所追求的目標，瑞昱亦相當重視提升同仁的工作生活平衡與健康認知，因此長年關注與持續規劃執行涵蓋全體同仁 (包含約聘同仁) 之相關促進方案。例如：建置各類社團、設置瑞昱元氣活力館、悅讀小棧及悅讀館圖書室、舉辦各項團隊創新活力的精彩活動與運動賽事、開設多元主題的瑞昱知性生活系列講座，以及定期提供健康促進與疾病預防相關講座與資訊分享。在同仁的生活福利方面，公司亦逐年規劃多元方案照護同仁健康

[Timeline FeatureV2 sample]
[ESG=UNK] [NUM=0] [YEAR=0] [PERCENT=0] [VAGUE=4] [VAGUE_HIGH=1] [CLEAR=1] [VT_ALREADY=0] [VT_SHORT=0] [VT_MID=0] [VT_LONG=0] [RAW_EVD=None] 工作、健康與家庭生活的平衡是多數人所追求的目標，瑞昱亦相當重視提升同仁的工作生活平衡與健康認知，因此長年關注與持續規劃執行涵蓋全體同仁 (包含約聘同仁) 之相關促進方案。例如：建置各類社團、設置瑞昱元氣活力館、悅讀小棧及悅讀館圖書室、舉辦各項團隊創新活力的精彩活動與運動賽事、開設多元主題的瑞昱知性生活系列講座，以及定期提供健康促進與疾病預防相關講座與資訊分享。在同仁的生活福利方面，公司亦逐年規劃多元方案照護同仁健康和提昇工作士氣、創造健康活力工作的職場文化，打造企業永續經營的健康體質與長遠競爭力。

[Timeline TestSafe sample]
[COMPANY=rt] [TICKER=2379] [PAGE=136] [NUM=0] [YEAR=0] [PERCENT=0] [TARGET_YEAR=-1] [YEAR_DIFF=-1] [YEAR_BUCKET=UNK] [HAS_COMPLETED=0] [HAS_ANNUAL=1] [HAS_CONTINUE=1] 

In [ ]:
expert_test_loaders_v2 = {
    "anchor": anchor_test_loader,
    "quality": quality_test_loader,
    "timeline": timeline_test_loader_v2
}

expert_test_loaders_testsafe = {
    "anchor": anchor_test_loader,
    "quality": quality_test_loader,
    "timeline": timeline_test_loader_testsafe
}

In [ ]:
# =====================================================
# 3. 用最佳設定跑 test
# =====================================================

test_predictions = expert_inference_quality_vt_ensemble(
    anchor_models=anchor_models,
    quality_models_fgm=quality_models_fgm,
    quality_models_nofgm=quality_models_nofgm,
    timeline_models_v2=timeline_models_v2,
    timeline_models_testsafe=timeline_models_testsafe,
    loaders_v2=expert_test_loaders_v2,
    loaders_testsafe=expert_test_loaders_testsafe,
    device=device,
    id2label=expert_id2label,
    no_threshold=None,
    es_fgm_weight=0.6,
    eq_fgm_weight=0.4,
    vt_v2_weight=0.4
)

print("Pred 筆數:", len(test_predictions))
print(test_predictions[:5])

Pred 筆數: 2000
[{'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'between_2_and_5_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'between_2_and_5_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}]


In [ ]:
from collections import Counter

print(Counter(x["verification_timeline"] for x in preds))

NameError: name 'preds' is not defined

In [ ]:
print(test_data[0].keys())
print(test_data[0])

dict_keys(['id', 'data', 'company', 'ticker', 'page_number', 'pdf_url', 'company_source'])
{'id': '12001', 'data': '工作、健康與家庭生活的平衡是多數人所追求的目標，瑞昱亦相當重視提升同仁的工作生活平衡與健康認知，因此長年關注與持續規劃執行涵蓋全體同仁 (包含約聘同仁) 之相關促進方案。例如：建置各類社團、設置瑞昱元氣活力館、悅讀小棧及悅讀館圖書室、舉辦各項團隊創新活力的精彩活動與運動賽事、開設多元主題的瑞昱知性生活系列講座，以及定期提供健康促進與疾病預防相關講座與資訊分享。在同仁的生活福利方面，公司亦逐年規劃多元方案照護同仁健康和提昇工作士氣、創造健康活力工作的職場文化，打造企業永續經營的健康體質與長遠競爭力。', 'company': 'rt', 'ticker': '2379', 'page_number': '136', 'pdf_url': 'https://www.realtek.com/images/csr/2024.pdf', 'company_source': 'https://www.realtek.com/Article/Index?menu_id=912&lang=zh-TW'}


In [ ]:
vt_preds = predict(model_timeline, timeline_test_loader, device, expert_id2label, ["verification_timeline"])
print(vt_preds[:5])  # 直接印出來看,是不是真的有非N/A的值

[{'verification_timeline': 'N/A'}, {'verification_timeline': 'N/A'}, {'verification_timeline': 'N/A'}, {'verification_timeline': 'N/A'}, {'verification_timeline': 'N/A'}]


In [ ]:
print(test_predictions[:10])

from collections import Counter

print(Counter(x["verification_timeline"] for x in test_predictions))
print(Counter(x["promise_status"] for x in test_predictions))

[{'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'between_2_and_5_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'between_2_and_5_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'already', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_status': 'Yes', 'verification_timeline': 'more_than_5_years', 'evidence_status': 'No', 'evidence_quality': 'N/A'}, {'promise_status': 'Yes', 'verification_timeline': 'more_than_5_years', 'evidence_status': 'Yes', 'evidence_quality': 'Clear'}, {'promise_

In [ ]:
# =====================================================
# 4. 轉成 submission 格式 + 強制修正邏輯規則
# =====================================================

def apply_submission_rules(preds):
    fixed = []

    for p in preds:
        p = p.copy()

        # label 名稱修正
       # if p["verification_timeline"] == "longer_than_5_years":
        #    p["verification_timeline"] = "more_than_5_years"

        # Rule 1: 非承諾段落
        if p["promise_status"] == "No":
            p["verification_timeline"] = "N/A"
            p["evidence_status"] = "N/A"
            p["evidence_quality"] = "N/A"

        # Rule 2: 承諾但無佐證
        if p["promise_status"] == "Yes" and p["evidence_status"] == "No":
            p["evidence_quality"] = "N/A"

        # Rule 3: evidence_status = N/A 時，EQ 也要 N/A
        if p["evidence_status"] == "N/A":
            p["evidence_quality"] = "N/A"

        fixed.append(p)

    return fixed


test_predictions_fixed = apply_submission_rules(test_predictions)

rows = []

for item, pred in zip(test_data, test_predictions_fixed):

    rows.append({
        "id": str(item["id"]),
        "promise_status": pred["promise_status"],
        "verification_timeline": pred["verification_timeline"],
        "evidence_status": pred["evidence_status"],
        "evidence_quality": pred["evidence_quality"]
    })

submission_df = pd.DataFrame(
    rows,
    columns=[
        "id",
        "promise_status",
        "verification_timeline",
        "evidence_status",
        "evidence_quality"
    ]
)

print(submission_df.head())




      id promise_status  verification_timeline evidence_status  \
0  12001            Yes                already             Yes   
1  12002            Yes  between_2_and_5_years             Yes   
2  12003            Yes                already             Yes   
3  12004            Yes  between_2_and_5_years             Yes   
4  12005            Yes                already             Yes   

  evidence_quality  
0            Clear  
1            Clear  
2            Clear  
3            Clear  
4            Clear  


In [ ]:
# =====================================================
# 5. 檢查 submission 是否符合規定
# =====================================================

allowed_ps = {"Yes", "No"}
allowed_vt = {
    "already",
    "within_2_years",
    "between_2_and_5_years",
    "more_than_5_years",
    "N/A"
}
allowed_es = {"Yes", "No", "N/A"}
allowed_eq = {"Clear", "Not Clear", "Misleading", "N/A"}

print("筆數:", len(submission_df))
print("欄位:", list(submission_df.columns))
print("NaN 數量:")
print(submission_df.isna().sum())

assert len(submission_df) == 2000

assert list(submission_df.columns) == [
    "id",
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality"
]

assert submission_df["id"].tolist() == [
    str(i) for i in range(12001, 14001)
]

assert set(submission_df["promise_status"]).issubset(allowed_ps)
assert set(submission_df["verification_timeline"]).issubset(allowed_vt)
assert set(submission_df["evidence_status"]).issubset(allowed_es)
assert set(submission_df["evidence_quality"]).issubset(allowed_eq)

# 邏輯規則檢查
bad_rows = []

for idx, row in submission_df.iterrows():

    if row["promise_status"] == "No":
        if not (
            row["verification_timeline"] == "N/A"
            and row["evidence_status"] == "N/A"
            and row["evidence_quality"] == "N/A"
        ):
            bad_rows.append(idx)

    if row["promise_status"] == "Yes" and row["evidence_status"] == "No":
        if row["evidence_quality"] != "N/A":
            bad_rows.append(idx)

    if row["evidence_status"] == "N/A":
        if row["evidence_quality"] != "N/A":
            bad_rows.append(idx)

print("邏輯錯誤筆數:", len(bad_rows))

assert len(bad_rows) == 0

print("✅ submission 格式檢查通過")
# =====================================================
# 6. Save Submission
# =====================================================

import os

OUTPUT_DIR = "submission"
os.makedirs(OUTPUT_DIR, exist_ok=True)

save_path = os.path.join(
    OUTPUT_DIR,
    "submission.csv"
)

submission_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8"
)

print(f"✅ Submission saved to: {save_path}")
print(submission_df.head(10))


# =====================================================
# 7. 看預測分布
# =====================================================

print("\nPromise Status 分布")
print(submission_df["promise_status"].value_counts())

print("\nVerification Timeline 分布")
print(submission_df["verification_timeline"].value_counts())

print("\nEvidence Status 分布")
print(submission_df["evidence_status"].value_counts())

print("\nEvidence Quality 分布")
print(submission_df["evidence_quality"].value_counts())

筆數: 2000
欄位: ['id', 'promise_status', 'verification_timeline', 'evidence_status', 'evidence_quality']
NaN 數量:
id                       0
promise_status           0
verification_timeline    0
evidence_status          0
evidence_quality         0
dtype: int64
邏輯錯誤筆數: 0
✅ submission 格式檢查通過
✅ 已輸出: /content/drive/MyDrive/AI_CUP_ESG/train2000vt0.45_eseqfgm0.6essemble_5fold_VTmore.csv
      id promise_status  verification_timeline evidence_status  \
0  12001            Yes                already             Yes   
1  12002            Yes  between_2_and_5_years             Yes   
2  12003            Yes                already             Yes   
3  12004            Yes  between_2_and_5_years             Yes   
4  12005            Yes                already             Yes   
5  12006            Yes                already             Yes   
6  12007            Yes      more_than_5_years              No   
7  12008            Yes      more_than_5_years             Yes   
8  12009            Yes  

In [ ]:
##驗證時可以 no gate，但正式提交一定要修規則：
def apply_submission_rules(preds):
   fixed = []

   for p in preds:
       p = p.copy()

       if p["verification_timeline"] == "longer_than_5_years":
           p["verification_timeline"] = "more_than_5_years"

       if p["promise_status"] == "No":
           p["verification_timeline"] = "N/A"
           p["evidence_status"] = "N/A"
           p["evidence_quality"] = "N/A"

       if p["promise_status"] == "Yes" and p["evidence_status"] == "No":
           p["evidence_quality"] = "N/A"

       fixed.append(p)

   return fixed



In [ ]:
test_preds_5fold = expert_inference_5fold(
    anchor_models,
    quality_models,
    timeline_models,
    expert_test_loaders,
    device,
    expert_id2label,
    no_threshold=0.6
)

test_preds_5fold_fixed = apply_submission_rules(
    test_preds_5fold
)



## Step 8: 用FGM！



In [ ]:
quality_task_weights = {
    "evidence_status": 0.2,
    "evidence_quality": 0.8
}

In [ ]:
# =========================
# FGM
# =========================
EPOCHS = 12
class FGM:
    def __init__(self, model, emb_name="word_embeddings", epsilon=1.0):
        self.model = model
        self.emb_name = emb_name
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()

                norm = torch.norm(param.grad)

                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]

        self.backup = {}


# =========================
# Train epoch (with exclude fields for adv loss)
# =========================
def train_expert_epoch_fgm(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    criterions,
    task_weights,
    alpha=1,
    fgm_epsilon=1.0,
    fgm_exclude_fields=None,
):
    model.train()
    fgm = FGM(model, epsilon=fgm_epsilon)
    fgm_exclude_fields = fgm_exclude_fields or []

    total_loss = 0.0

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = {
            k: v.to(device)
            for k, v in batch["labels"].items()
        }

        optimizer.zero_grad()

        # =========================
        # Normal forward
        # =========================
        outputs = model(input_ids, attention_mask)

        loss = 0
        for field, logits in outputs.items():
            if field in labels:
                loss_task = criterions[field](logits, labels[field])
                loss += task_weights[field] * loss_task

        loss.backward()

        # =========================
        # FGM adversarial forward
        # =========================
        fgm.attack()

        outputs_adv = model(input_ids, attention_mask)

        loss_adv = 0
        for field, logits in outputs_adv.items():
            if field in labels and field not in fgm_exclude_fields:
                loss_task_adv = criterions[field](logits, labels[field])
                loss_adv += task_weights[field] * loss_task_adv

        if isinstance(loss_adv, torch.Tensor):
            loss_adv.backward()

        fgm.restore()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        loss_adv_val = loss_adv.item() if isinstance(loss_adv, torch.Tensor) else 0.0
        total_loss += (loss.item() + loss_adv_val)

    return total_loss / len(loader)


# =========================
# Run training (per fold)
# =========================
def run_quality_training_fgm(
    expert_name,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterions,
    task_weights,
    fields,
    save_path,
    device,
    alpha=0,
    fgm_epsilon=1.0,
    fgm_exclude_fields=None,
):
    print(f"\n🚀 開始訓練 {expert_name}！FGM epsilon={fgm_epsilon}, exclude={fgm_exclude_fields}")
    print("=" * 60)

    best_score = 0.0

    for epoch in range(EPOCHS):

        print(f"\n[{expert_name}] Epoch {epoch+1}/{EPOCHS}")
        print("-" * 40)

        avg_loss = train_expert_epoch_fgm(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            criterions=criterions,
            task_weights=task_weights,
            alpha=alpha,
            fgm_epsilon=fgm_epsilon,
            fgm_exclude_fields=fgm_exclude_fields,
        )

        val_preds_short = predict(
            model,
            val_loader,
            device,
            expert_id2label,
            fields
        )

        val_gt_full = []
        val_preds_full = []

        val_items = val_loader.dataset.data

        for i, item in enumerate(val_items):

            gt_item = {
                "promise_status": item.get("promise_status", "N/A"),
                "evidence_status": item.get("evidence_status", "N/A"),
                "evidence_quality": item.get("evidence_quality", "N/A"),
                "verification_timeline": item.get("verification_timeline", "N/A")
            }

            pred_item = {
                "promise_status": "N/A",
                "evidence_status": "N/A",
                "evidence_quality": "N/A",
                "verification_timeline": "N/A"
            }

            for f in fields:
                pred_item[f] = val_preds_short[i][f]

            val_gt_full.append(gt_item)
            val_preds_full.append(pred_item)

        results = evaluate_hybrid(
            val_gt_full,
            val_preds_full
        )

        current_score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in fields
        )

        print(f"  平均 Loss: {avg_loss:.4f} | 加權得分: {current_score:.5f}")

        for f in fields:
            print(f"  -> {f:22}: Macro F1 = {results[f]['macro_f1']:.4f}")

        if current_score > best_score:
            best_score = current_score
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ 最佳模型已儲存！得分: {best_score:.5f}")

    return best_score


# =========================
# Fold loop
# =========================
quality_fold_scores_fgm = []

for fold_id in range(5):

    print(f"\n========== Quality FGM Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_quality = QualitySharedExpert(expert_label_counts).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = f"{SAVE_DIR}/best_quality_es_fgm_v2_fold{fold_id+1}.pt"

    best_quality = run_quality_training_fgm(
        expert_name=f"Quality_FGM_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        device=device,
        alpha=1,
        fgm_epsilon=0.3,
        fgm_exclude_fields=["evidence_quality"],
    )

    quality_fold_scores_fgm.append({
        "fold": fold_id + 1,
        "quality_fgm": best_quality
    })

    print(f"✅ Quality FGM Fold {fold_id+1} done | Score: {best_quality:.5f}")


========== Quality FGM Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold1！FGM epsilon=0.3, exclude=['evidence_quality']

[Quality_FGM_Fold1] Epoch 1/12
----------------------------------------
  平均 Loss: 1.2377 | 加權得分: 0.27761
  -> evidence_status       : Macro F1 = 0.5012
  -> evidence_quality      : Macro F1 = 0.3636
  ✅ 最佳模型已儲存！得分: 0.27761

[Quality_FGM_Fold1] Epoch 2/12
----------------------------------------
  平均 Loss: 0.9919 | 加權得分: 0.30284
  -> evidence_status       : Macro F1 = 0.6012
  -> evidence_quality      : Macro F1 = 0.3499
  ✅ 最佳模型已儲存！得分: 0.30284

[Quality_FGM_Fold1] Epoch 3/12
----------------------------------------
  平均 Loss: 0.7459 | 加權得分: 0.32068
  -> evidence_status       : Macro F1 = 0.6078
  -> evidence_quality      : Macro F1 = 0.3953
  ✅ 最佳模型已儲存！得分: 0.32068

[Quality_FGM_Fold1] Epoch 4/12
----------------------------------------
  平均 Loss: 0.5340 | 加權得分: 0.32380
  -> evidence_status       : Macro F1 = 0.6109
  -> evidence_quality      : Macro F1 = 0.4015
  ✅ 最佳模型已儲存！得分: 0.32380

[Quality_FGM_Fold1] Epoch 5

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold4！FGM epsilon=0.3, exclude=['evidence_quality']

[Quality_FGM_Fold4] Epoch 1/12
----------------------------------------
  平均 Loss: 1.2581 | 加權得分: 0.30169
  -> evidence_status       : Macro F1 = 0.5804
  -> evidence_quality      : Macro F1 = 0.3644
  ✅ 最佳模型已儲存！得分: 0.30169

[Quality_FGM_Fold4] Epoch 2/12
----------------------------------------
  平均 Loss: 1.0497 | 加權得分: 0.34035
  -> evidence_status       : Macro F1 = 0.6574
  -> evidence_quality      : Macro F1 = 0.4090
  ✅ 最佳模型已儲存！得分: 0.34035

[Quality_FGM_Fold4] Epoch 3/12
----------------------------------------
  平均 Loss: 0.8159 | 加權得分: 0.35019
  -> evidence_status       : Macro F1 = 0.6055
  -> evidence_quality      : Macro F1 = 0.4815
  ✅ 最佳模型已儲存！得分: 0.35019

[Quality_FGM_Fold4] Epoch 4/12
----------------------------------------
  平均 Loss: 0.5560 | 加權得分: 0.34497
  -> evidence_status       : Macro F1 = 0.6303
  -> evidence_quality      : Macro F1 = 0.4453

[Quality_FGM_Fold4] Epoch 5/12
--------------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold5！FGM epsilon=0.3, exclude=['evidence_quality']

[Quality_FGM_Fold5] Epoch 1/12
----------------------------------------
  平均 Loss: 1.2633 | 加權得分: 0.27681
  -> evidence_status       : Macro F1 = 0.4859
  -> evidence_quality      : Macro F1 = 0.3744
  ✅ 最佳模型已儲存！得分: 0.27681

[Quality_FGM_Fold5] Epoch 2/12
----------------------------------------
  平均 Loss: 1.0485 | 加權得分: 0.34037
  -> evidence_status       : Macro F1 = 0.5998
  -> evidence_quality      : Macro F1 = 0.4584
  ✅ 最佳模型已儲存！得分: 0.34037

[Quality_FGM_Fold5] Epoch 3/12
----------------------------------------
  平均 Loss: 0.8206 | 加權得分: 0.35561
  -> evidence_status       : Macro F1 = 0.6647
  -> evidence_quality      : Macro F1 = 0.4463
  ✅ 最佳模型已儲存！得分: 0.35561

[Quality_FGM_Fold5] Epoch 4/12
----------------------------------------
  平均 Loss: 0.5614 | 加權得分: 0.35014
  -> evidence_status       : Macro F1 = 0.6523
  -> evidence_quality      : Macro F1 = 0.4413

[Quality_FGM_Fold5] Epoch 5/12
--------------------

##1. FGM class

In [ ]:

class FGM:
    def __init__(self, model, emb_name="word_embeddings", epsilon=0.5):
        self.model = model
        self.emb_name = emb_name
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()

                norm = torch.norm(param.grad)

                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]

        self.backup = {}

##2. 訓練 epoch：加 FGM 版

這版是給 Quality Expert 用的。

In [ ]:


def train_expert_epoch_fgm(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    criterions,
    task_weights,
    alpha=1,
    fgm_epsilon=1.0
):
    model.train()
    fgm = FGM(model, epsilon=fgm_epsilon)

    total_loss = 0.0

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = {
            k: v.to(device)
            for k, v in batch["labels"].items()
        }

        optimizer.zero_grad()

        # =========================
        # Normal forward
        # =========================
        outputs = model(input_ids, attention_mask)

        loss = 0

        for field, logits in outputs.items():
            if field in labels:
                loss_task = criterions[field](logits, labels[field])
                loss += task_weights[field] * loss_task

        loss.backward()

        # =========================
        # FGM adversarial forward
        # =========================
        fgm.attack()

        outputs_adv = model(input_ids, attention_mask)

        loss_adv = 0

        for field, logits in outputs_adv.items():
            if field in labels:
                loss_task_adv = criterions[field](logits, labels[field])
                loss_adv += task_weights[field] * loss_task_adv

        loss_adv.backward()

        fgm.restore()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += (loss.item() + loss_adv.item())

    return total_loss / len(loader)

##3. Quality 專用 training function

In [ ]:

def run_quality_training_fgm(
    expert_name,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterions,
    task_weights,
    fields,
    save_path,
    device,
    alpha=1,
    fgm_epsilon=1.0
):
    print(f"\n🚀 開始訓練 {expert_name}！FGM epsilon={fgm_epsilon}")
    print("=" * 60)

    best_score = 0.0

    for epoch in range(EPOCHS):

        print(f"\n[{expert_name}] Epoch {epoch+1}/{EPOCHS}")
        print("-" * 40)

        avg_loss = train_expert_epoch_fgm(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            criterions=criterions,
            task_weights=task_weights,
            alpha=alpha,
            fgm_epsilon=fgm_epsilon
        )

        val_preds_short = predict(
            model,
            val_loader,
            device,
            expert_id2label,
            fields
        )

        val_gt_full = []
        val_preds_full = []

        # 注意：這裡如果你是 5-fold，要改成外面傳入的 val_fold
        # 所以下面先用 val_loader.dataset.data
        val_items = val_loader.dataset.data

        for i, item in enumerate(val_items):

            gt_item = {
                "promise_status": item.get("promise_status", "N/A"),
                "evidence_status": item.get("evidence_status", "N/A"),
                "evidence_quality": item.get("evidence_quality", "N/A"),
                "verification_timeline": item.get("verification_timeline", "N/A")
            }

            pred_item = {
                "promise_status": "N/A",
                "evidence_status": "N/A",
                "evidence_quality": "N/A",
                "verification_timeline": "N/A"
            }

            for f in fields:
                pred_item[f] = val_preds_short[i][f]

            val_gt_full.append(gt_item)
            val_preds_full.append(pred_item)

        results = evaluate_hybrid(
            val_gt_full,
            val_preds_full
        )

        current_score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in fields
        )

        print(f"  平均 Loss: {avg_loss:.4f} | 加權得分: {current_score:.5f}")

        for f in fields:
            print(f"  -> {f:22}: Macro F1 = {results[f]['macro_f1']:.4f}")

        if current_score > best_score:
            best_score = current_score
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ 最佳模型已儲存！得分: {best_score:.5f}")

    return best_score

In [ ]:
# =====================================================
# FGM
# =====================================================

class FGM:
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1.0, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if (
                param.requires_grad
                and emb_name in name
                and param.grad is not None
            ):
                self.backup[name] = param.data.clone()

                norm = torch.norm(param.grad)

                if norm != 0:
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name="embeddings"):
        for name, param in self.model.named_parameters():
            if (
                param.requires_grad
                and emb_name in name
                and name in self.backup
            ):
                param.data = self.backup[name]

        self.backup = {}
# =====================================================
# Train one epoch with FGM exclude fields
# =====================================================

def train_expert_epoch_fgm(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    criterions,
    task_weights,
    alpha=1,
    fgm_epsilon=1.0,
    exclude_fgm_fields=None
):
    """
    exclude_fgm_fields:
      []                         -> 所有 task 都做 FGM
      ["evidence_quality"]       -> EQ 不做 FGM，只 ES 做
      ["evidence_status"]        -> ES 不做 FGM，只 EQ 做
      ["evidence_status", "evidence_quality"] -> 等於沒有 FGM
    """

    if exclude_fgm_fields is None:
        exclude_fgm_fields = []

    model.train()
    total_loss = 0.0

    fgm = FGM(model)

    for batch in loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = {
            k: v.to(device)
            for k, v in batch["labels"].items()
        }

        # =====================================================
        # 1. Normal loss：所有 task 都正常訓練
        # =====================================================

        outputs = model(
            input_ids,
            attention_mask
        )

        loss = 0.0

        for field in outputs:

            if field not in labels:
                continue

            task_loss = criterions[field](
                outputs[field],
                labels[field]
            )

            loss = loss + task_weights[field] * task_loss

        loss.backward()

        # =====================================================
        # 2. FGM adversarial loss：排除指定 task
        # =====================================================

        fgm.attack(
            epsilon=fgm_epsilon
        )

        outputs_adv = model(
            input_ids,
            attention_mask
        )

        adv_loss = 0.0
        used_adv_task = False

        for field in outputs_adv:

            if field not in labels:
                continue

            if field in exclude_fgm_fields:
                continue

            task_loss = criterions[field](
                outputs_adv[field],
                labels[field]
            )

            adv_loss = adv_loss + task_weights[field] * task_loss
            used_adv_task = True

        if used_adv_task:
            adv_loss.backward()

        fgm.restore()

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)
# =====================================================
# Run Quality Training FGM with exclude fields
# =====================================================

def run_quality_training_fgm(
    expert_name,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterions,
    task_weights,
    fields,
    save_path,
    device,
    alpha=1,
    fgm_epsilon=1.0,
    exclude_fgm_fields=None
):
    if exclude_fgm_fields is None:
        exclude_fgm_fields = []

    print(f"\n🚀 開始訓練 {expert_name}")
    print(f"FGM epsilon={fgm_epsilon}")
    print(f"exclude_fgm_fields={exclude_fgm_fields}")
    print("=" * 60)

    best_score = 0.0

    for epoch in range(EPOCHS):

        print(f"\n[{expert_name}] Epoch {epoch+1}/{EPOCHS}")
        print("-" * 40)

        avg_loss = train_expert_epoch_fgm(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            criterions=criterions,
            task_weights=task_weights,
            alpha=alpha,
            fgm_epsilon=fgm_epsilon,
            exclude_fgm_fields=exclude_fgm_fields
        )

        val_preds_short = predict(
            model,
            val_loader,
            device,
            expert_id2label,
            fields
        )

        val_items = val_loader.dataset.data

        val_gt_full = []
        val_preds_full = []

        for i, item in enumerate(val_items):

            gt_item = {
                "promise_status": item.get("promise_status", "N/A"),
                "evidence_status": item.get("evidence_status", "N/A"),
                "evidence_quality": item.get("evidence_quality", "N/A"),
                "verification_timeline": item.get("verification_timeline", "N/A")
            }

            pred_item = {
                "promise_status": "N/A",
                "evidence_status": "N/A",
                "evidence_quality": "N/A",
                "verification_timeline": "N/A"
            }

            for f in fields:
                pred_item[f] = val_preds_short[i][f]

            val_gt_full.append(gt_item)
            val_preds_full.append(pred_item)

        results = evaluate_hybrid(
            val_gt_full,
            val_preds_full
        )

        current_score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in fields
        )

        print(f"  平均 Loss: {avg_loss:.4f} | 加權得分: {current_score:.5f}")

        for f in fields:
            print(f"  -> {f:22}: Macro F1 = {results[f]['macro_f1']:.4f}")

        if current_score > best_score:
            best_score = current_score
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ 最佳模型已儲存！得分: {best_score:.5f}")

    return best_score



In [ ]:
def run_quality_training_eqfgm(
    expert_name,
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    criterions,
    task_weights,
    fields,
    save_path,
    device,
    alpha=1,
    fgm_epsilon=1.0,
    exclude_fgm_fields=None
):
    if exclude_fgm_fields is None:
        exclude_fgm_fields = []

    print(f"\n🚀 開始訓練 {expert_name}")
    print(f"FGM epsilon={fgm_epsilon}")
    print(f"exclude_fgm_fields={exclude_fgm_fields}")
    print("=" * 60)

    best_score = 0.0

    for epoch in range(EPOCHS):

        print(f"\n[{expert_name}] Epoch {epoch+1}/{EPOCHS}")
        print("-" * 40)

        avg_loss = train_expert_epoch_fgm(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            criterions=criterions,
            task_weights=task_weights,
            alpha=alpha,
            fgm_epsilon=fgm_epsilon,
            exclude_fgm_fields=exclude_fgm_fields
        )

        val_preds_short = predict(
            model,
            val_loader,
            device,
            expert_id2label,
            fields
        )

        val_items = val_loader.dataset.data

        val_gt_full = []
        val_preds_full = []

        for i, item in enumerate(val_items):

            gt_item = {
                "promise_status": item.get("promise_status", "N/A"),
                "evidence_status": item.get("evidence_status", "N/A"),
                "evidence_quality": item.get("evidence_quality", "N/A"),
                "verification_timeline": item.get("verification_timeline", "N/A")
            }

            pred_item = {
                "promise_status": "N/A",
                "evidence_status": "N/A",
                "evidence_quality": "N/A",
                "verification_timeline": "N/A"
            }

            for f in fields:
                pred_item[f] = val_preds_short[i][f]

            val_gt_full.append(gt_item)
            val_preds_full.append(pred_item)

        results = evaluate_hybrid(
            val_gt_full,
            val_preds_full
        )

        current_score = sum(
            results[f]["macro_f1"] * FIELD_WEIGHTS[f]
            for f in fields
        )

        print(f"  平均 Loss: {avg_loss:.4f} | 加權得分: {current_score:.5f}")

        for f in fields:
            print(f"  -> {f:22}: Macro F1 = {results[f]['macro_f1']:.4f}")

        eq_score = results["evidence_quality"]["macro_f1"]

        if eq_score > best_score:
            best_score = eq_score

            torch.save(
                model.state_dict(),
                save_path
            )

            print(
                f"  ✅ Best EQ saved: {best_score:.5f}"
            )

    return best_score

##4. 在 5-fold Quality 裡使用 FGM

把 Quality 那段換成這樣：

In [ ]:
EPOCHS = 12

# =====================================================
# Quality FGM 5-fold
# ES 做 FGM，EQ 不做 FGM
# =====================================================

quality_fold_scores_fgm = []

for fold_id in range(5):

    print(f"\n========== Quality FGM Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]

    loaders = build_fold_loaders(
        train_fold,
        val_fold
    )

    model_quality = QualitySharedExpert(
        expert_label_counts
    ).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = (
        f"{SAVE_DIR}/best_quality_fgm_esonly_fold{fold_id+1}.pt"
    )

    best_quality = run_quality_training_fgm(
        expert_name=f"Quality_FGM_ESOnly_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        device=device,
        alpha=1,
        fgm_epsilon=0.5,
        exclude_fgm_fields=[
            "evidence_quality"
        ]
    )

    quality_fold_scores_fgm.append({
        "fold": fold_id + 1,
        "quality_fgm_esonly": best_quality
    })

    print(
        f"✅ Quality FGM ESOnly Fold {fold_id+1} done | "
        f"Score: {best_quality:.5f}"
    )

quality_score_df_fgm = pd.DataFrame(
    quality_fold_scores_fgm
)

print(quality_score_df_fgm)

print(
    "Quality FGM ESOnly Mean:",
    quality_score_df_fgm["quality_fgm_esonly"].mean()
)


========== Quality FGM Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_ESOnly_Fold1
FGM epsilon=0.5
exclude_fgm_fields=['evidence_quality']

[Quality_FGM_ESOnly_Fold1] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0677 | 加權得分: 0.31755
  -> evidence_status       : Macro F1 = 0.5627
  -> evidence_quality      : Macro F1 = 0.4250
  ✅ 最佳模型已儲存！得分: 0.31755

[Quality_FGM_ESOnly_Fold1] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8899 | 加權得分: 0.34923
  -> evidence_status       : Macro F1 = 0.6476
  -> evidence_quality      : Macro F1 = 0.4427
  ✅ 最佳模型已儲存！得分: 0.34923

[Quality_FGM_ESOnly_Fold1] Epoch 3/12
----------------------------------------
  平均 Loss: 0.7138 | 加權得分: 0.34590
  -> evidence_status       : Macro F1 = 0.6371
  -> evidence_quality      : Macro F1 = 0.4422

[Quality_FGM_ESOnly_Fold1] Epoch 4/12
----------------------------------------
  平均 Loss: 0.5108 | 加權得分: 0.35718
  -> evidence_status       : Macro F1 = 0.6559
  -> evidence_quality      : Macro F1 = 0.4583
  ✅ 最佳模型已儲存！得分: 0.35718

[Quali

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_ESOnly_Fold4
FGM epsilon=0.5
exclude_fgm_fields=['evidence_quality']

[Quality_FGM_ESOnly_Fold4] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0251 | 加權得分: 0.35047
  -> evidence_status       : Macro F1 = 0.6604
  -> evidence_quality      : Macro F1 = 0.4353
  ✅ 最佳模型已儲存！得分: 0.35047

[Quality_FGM_ESOnly_Fold4] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8319 | 加權得分: 0.34918
  -> evidence_status       : Macro F1 = 0.6457
  -> evidence_quality      : Macro F1 = 0.4442

[Quality_FGM_ESOnly_Fold4] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6364 | 加權得分: 0.37614
  -> evidence_status       : Macro F1 = 0.6971
  -> evidence_quality      : Macro F1 = 0.4772
  ✅ 最佳模型已儲存！得分: 0.37614

[Quality_FGM_ESOnly_Fold4] Epoch 4/12
----------------------------------------
  平均 Loss: 0.4699 | 加權得分: 0.38082
  -> evidence_status       : Macro F1 = 0.7267
  -> evidence_quality      : Macro F1 = 0.4651
  ✅ 最佳模型已儲存！得分: 0.38082

[Quali

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_ESOnly_Fold5
FGM epsilon=0.5
exclude_fgm_fields=['evidence_quality']

[Quality_FGM_ESOnly_Fold5] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0102 | 加權得分: 0.29608
  -> evidence_status       : Macro F1 = 0.5309
  -> evidence_quality      : Macro F1 = 0.3909
  ✅ 最佳模型已儲存！得分: 0.29608

[Quality_FGM_ESOnly_Fold5] Epoch 2/12
----------------------------------------
  平均 Loss: 0.7899 | 加權得分: 0.34955
  -> evidence_status       : Macro F1 = 0.6687
  -> evidence_quality      : Macro F1 = 0.4255
  ✅ 最佳模型已儲存！得分: 0.34955

[Quality_FGM_ESOnly_Fold5] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6389 | 加權得分: 0.35130
  -> evidence_status       : Macro F1 = 0.6973
  -> evidence_quality      : Macro F1 = 0.4060
  ✅ 最佳模型已儲存！得分: 0.35130

[Quality_FGM_ESOnly_Fold5] Epoch 4/12
----------------------------------------
  平均 Loss: 0.4606 | 加權得分: 0.36236
  -> evidence_status       : Macro F1 = 0.6780
  -> evidence_quality      : Macro F1 = 0.4542
  ✅ 最佳模

In [ ]:
def predict(model, dataloader, device, id2label, fields):
    """
    通用專家預測函式：
    將 model 切換至 eval 模式，並根據 fields 產出預測字典。
    """
    model.eval()
    all_preds = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # 取得 logits
            logits = model(input_ids, attention_mask)

            batch_size = input_ids.size(0)
            for i in range(batch_size):
                pred_item = {}
                for field in fields:
                    # 找到該 field 最大機率的 ID 並轉回 Label 文字
                    pred_id = logits[field][i].argmax().item()
                    pred_item[field] = id2label[field][pred_id]
                all_preds.append(pred_item)

    return all_preds


In [ ]:
EPOCHS = 12

# =====================================================
# Quality FGM 5-fold
# EQ 做 FGM，ES 不做 FGM
# =====================================================

quality_fold_scores_fgm = []

for fold_id in range(4,5):

    print(f"\n========== Quality FGM Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]

    loaders = build_fold_loaders(
        train_fold,
        val_fold
    )

    model_quality = QualitySharedExpert(
        expert_label_counts
    ).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = (
        f"{SAVE_DIR}/best_quality_fgm_eqonly_fold{fold_id+1}.pt"
    )

    best_quality = run_quality_training_eqfgm(
        expert_name=f"Quality_FGM_EQOnly_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        device=device,
        alpha=1,
        fgm_epsilon=0.3,
        exclude_fgm_fields=[
            "evidence_status"
        ]
    )

    quality_fold_scores_fgm.append({
        "fold": fold_id + 1,
        "quality_fgm_eqonly": best_quality
    })

    print(
        f"✅ Quality FGM EQOnly Fold {fold_id+1} done | "
        f"Score: {best_quality:.5f}"
    )

quality_score_df_fgm = pd.DataFrame(
    quality_fold_scores_fgm
)

print(quality_score_df_fgm)

print(
    "Quality FGM ESOnly Mean:",
    quality_score_df_fgm["quality_fgm_eqonly"].mean()
)


========== Quality FGM Fold 5/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_EQOnly_Fold5
FGM epsilon=0.3
exclude_fgm_fields=['evidence_status']

[Quality_FGM_EQOnly_Fold5] Epoch 1/12
----------------------------------------
  平均 Loss: 1.0430 | 加權得分: 0.29281
  -> evidence_status       : Macro F1 = 0.5423
  -> evidence_quality      : Macro F1 = 0.3718
  ✅ Best EQ saved: 0.37180

[Quality_FGM_EQOnly_Fold5] Epoch 2/12
----------------------------------------
  平均 Loss: 0.8588 | 加權得分: 0.34185
  -> evidence_status       : Macro F1 = 0.6423
  -> evidence_quality      : Macro F1 = 0.4262
  ✅ Best EQ saved: 0.42616

[Quality_FGM_EQOnly_Fold5] Epoch 3/12
----------------------------------------
  平均 Loss: 0.6937 | 加權得分: 0.34598
  -> evidence_status       : Macro F1 = 0.6832
  -> evidence_quality      : Macro F1 = 0.4029

[Quality_FGM_EQOnly_Fold5] Epoch 4/12
----------------------------------------
  平均 Loss: 0.5091 | 加權得分: 0.36012
  -> evidence_status       : Macro F1 = 0.6872
  -> evidence_quality      : Macro F1 = 0.4399
  ✅ Best EQ saved: 0.43986

KeyboardInterrupt: 

In [ ]:

quality_fold_scores_fgm = []

for fold_id in range(5):

    print(f"\n========== Quality FGM Fold {fold_id + 1}/5 ==========")

    train_fold, val_fold = folds[fold_id]
    loaders = build_fold_loaders(train_fold, val_fold)

    model_quality = QualitySharedExpert(expert_label_counts).to(device)

    optimizer_quality = AdamW(
        model_quality.parameters(),
        lr=2e-5
    )

    scheduler_quality = get_linear_schedule_with_warmup(
        optimizer_quality,
        num_warmup_steps=0,
        num_training_steps=len(loaders["quality_train"]) * EPOCHS
    )

    quality_save_path = f"{SAVE_DIR}/best_quality_fgm_fold{fold_id+1}.pt"

    best_quality = run_quality_training_fgm(
        expert_name=f"Quality_FGM_Fold{fold_id+1}",
        model=model_quality,
        train_loader=loaders["quality_train"],
        val_loader=loaders["quality_val"],
        optimizer=optimizer_quality,
        scheduler=scheduler_quality,
        criterions=quality_criterions,
        task_weights=quality_task_weights,
        fields=quality_fields,
        save_path=quality_save_path,
        device=device,
        alpha=1,
        fgm_epsilon=1.0
    )

    quality_fold_scores_fgm.append({
        "fold": fold_id + 1,
        "quality_fgm": best_quality
    })

    print(f"✅ Quality FGM Fold {fold_id+1} done | Score: {best_quality:.5f}")


========== Quality FGM Fold 1/5 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold1！FGM epsilon=1.0

[Quality_FGM_Fold1] Epoch 1/10
----------------------------------------
  平均 Loss: 4.7042 | 加權得分: 0.28109
  -> evidence_status       : Macro F1 = 0.5611
  -> evidence_quality      : Macro F1 = 0.3222
  ✅ 最佳模型已儲存！得分: 0.28109

[Quality_FGM_Fold1] Epoch 2/10
----------------------------------------
  平均 Loss: 3.9273 | 加權得分: 0.32096
  -> evidence_status       : Macro F1 = 0.5954
  -> evidence_quality      : Macro F1 = 0.4067
  ✅ 最佳模型已儲存！得分: 0.32096

[Quality_FGM_Fold1] Epoch 3/10
----------------------------------------
  平均 Loss: 3.2574 | 加權得分: 0.33813
  -> evidence_status       : Macro F1 = 0.6246
  -> evidence_quality      : Macro F1 = 0.4307
  ✅ 最佳模型已儲存！得分: 0.33813

[Quality_FGM_Fold1] Epoch 4/10
----------------------------------------
  平均 Loss: 2.5048 | 加權得分: 0.35872
  -> evidence_status       : Macro F1 = 0.6852
  -> evidence_quality      : Macro F1 = 0.4376
  ✅ 最佳模型已儲存！得分: 0.35872

[Quality_FGM_Fold1] Epoch 5/10
--------------------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold4！FGM epsilon=1.0

[Quality_FGM_Fold4] Epoch 1/10
----------------------------------------
  平均 Loss: 4.6937 | 加權得分: 0.32809
  -> evidence_status       : Macro F1 = 0.6255
  -> evidence_quality      : Macro F1 = 0.4013
  ✅ 最佳模型已儲存！得分: 0.32809

[Quality_FGM_Fold4] Epoch 2/10
----------------------------------------
  平均 Loss: 3.9848 | 加權得分: 0.33977
  -> evidence_status       : Macro F1 = 0.6314
  -> evidence_quality      : Macro F1 = 0.4296
  ✅ 最佳模型已儲存！得分: 0.33977

[Quality_FGM_Fold4] Epoch 3/10
----------------------------------------
  平均 Loss: 3.2787 | 加權得分: 0.35528
  -> evidence_status       : Macro F1 = 0.6423
  -> evidence_quality      : Macro F1 = 0.4645
  ✅ 最佳模型已儲存！得分: 0.35528

[Quality_FGM_Fold4] Epoch 4/10
----------------------------------------
  平均 Loss: 2.5390 | 加權得分: 0.36402
  -> evidence_status       : Macro F1 = 0.6617
  -> evidence_quality      : Macro F1 = 0.4728
  ✅ 最佳模型已儲存！得分: 0.36402

[Quality_FGM_Fold4] Epoch 5/10
--------------------------

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🚀 開始訓練 Quality_FGM_Fold5！FGM epsilon=1.0

[Quality_FGM_Fold5] Epoch 1/10
----------------------------------------
  平均 Loss: 4.6720 | 加權得分: 0.27261
  -> evidence_status       : Macro F1 = 0.5782
  -> evidence_quality      : Macro F1 = 0.2833
  ✅ 最佳模型已儲存！得分: 0.27261

[Quality_FGM_Fold5] Epoch 2/10
----------------------------------------
  平均 Loss: 3.8668 | 加權得分: 0.29413
  -> evidence_status       : Macro F1 = 0.5613
  -> evidence_quality      : Macro F1 = 0.3592
  ✅ 最佳模型已儲存！得分: 0.29413

[Quality_FGM_Fold5] Epoch 3/10
----------------------------------------
  平均 Loss: 3.1682 | 加權得分: 0.33532
  -> evidence_status       : Macro F1 = 0.5830
  -> evidence_quality      : Macro F1 = 0.4583
  ✅ 最佳模型已儲存！得分: 0.33532

[Quality_FGM_Fold5] Epoch 4/10
----------------------------------------
  平均 Loss: 2.4972 | 加權得分: 0.35711
  -> evidence_status       : Macro F1 = 0.6515
  -> evidence_quality      : Macro F1 = 0.4619
  ✅ 最佳模型已儲存！得分: 0.35711

[Quality_FGM_Fold5] Epoch 5/10
--------------------------

## Step 8: 用test資料集！

